# 03 — Limpieza, variables y ruta de inferencia

**Objetivo:** convertir el `master_matches.parquet` (1 fila/partido, mezcla de señales
*pre-partido* y *post-partido*) en las features que consume el modelo, para **los dos casos**
que hacen falta en un predictor:

- **Entrenamiento** → partidos ya jugados, con resultado conocido → `features_matches.parquet`.
- **Inferencia (scoring)** → partidos futuros a predecir, sin resultado → `features_upcoming.parquet`.

La idea que vertebra todo el notebook: **es el MISMO código de features para los dos casos.** Primero
se construye y se audita la maquinaria de features sobre los partidos jugados (Parte A). Después, la
inferencia (Parte B) **reutiliza esa misma maquinaria** sobre los partidos futuros, sin reimplementar
nada. Por eso van juntos y en este orden: no se puede puntuar hasta que la maquinaria de
entrenamiento existe y está validada.

Dos principios que gobiernan las dos partes:
1. **Solo entra al modelo lo que se conoce antes del pitido inicial.** Las estadísticas del propio
   partido (goles, tiros, xG, rating...) se usan **solo** como materia prima para construir *forma* de
   partidos anteriores, siempre con `shift()`.
2. **La feature de cualquier partido — jugado o futuro — solo puede mirar hacia atrás.** Este es el
   mismo principio que hace el entrenamiento leak-safe y el que permite reutilizarlo para el futuro.

## 0. Dónde encaja este notebook

Este es el punto donde el proyecto deja de *recolectar y ordenar datos* y empieza a *fabricar lo que el
modelo realmente come*. El pipeline tiene tres tramos:

- **01 · Extracción** — cada fuente (football-data, Understat, ClubElo, SofaScore, Transfermarkt) se
  baja y se normaliza a un parquet en `data/02_processed/`.
- **02 · Consolidación** — esos parquets se cosen en un único `master_matches.parquet`: una fila por
  partido jugado, mezclando lo de *antes* del partido y lo de *después*.
- **03 · Este notebook** — coge ese máster y produce las *features*: las variables limpias, sin fugas y
  comparables que entran al modelo.

**Por qué entrenamiento e inferencia van en el mismo notebook.** Un predictor fabrica features en dos
momentos —cuando **aprende** (miles de partidos jugados, con resultado) y cuando **predice** (los pocos
partidos de la próxima jornada, sin jugar)—. La trampa clásica es escribir esa lógica *dos veces* y que
se desincronicen. Aquí no puede pasar: **hay un solo motor de features (`build_features`) y las dos
partes lo comparten.** La Parte A lo define y lo audita sobre lo jugado; la Parte B lo reutiliza *tal
cual* sobre lo por jugar. De ahí el orden —A antes que B, porque la inferencia importa `build_features`—
y que nunca corran a la vez.

**El principio que lo hace posible.** La feature de cualquier partido —jugado o futuro— solo puede mirar
**hacia atrás**: las estadísticas del propio partido (goles, tiros, xG, rating...) nunca entran crudas,
solo como materia prima de la *forma* de partidos anteriores, siempre con `shift()`. Un partido futuro
no es más que un partido cuyo "hacia atrás" ya existe y cuyo "hacia delante" está en blanco.

**Un aviso sobre la Parte B.** Entrenar es cómodo: el pasado no cambia. Predecir el futuro real es
operar sobre un calendario *vivo* —partidos que se aplazan, cuotas aún sin publicar, alineaciones que no
se conocen hasta última hora—. Por eso la Parte B añade comprobaciones operativas (¿están todos los
partidos de la jornada?, ¿hay cuota para cada uno?) que en el entrenamiento no hacen falta.

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
for p in (ROOT/'scripts'/'features', ROOT/'scripts'/'consolidate', ROOT/'scripts'/'extractors'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 60)

# Parte A (entrenamiento) - una familia de features = un modulo
import columns as C
from teamlong import to_team_long
from form import build_form_features, form_feature_names
from market_form import build_market_form_features, market_form_feature_names
from h2h import build_h2h_features, h2h_feature_names
from standings import build_standings_features, standings_feature_names
from prematch_derived import build_prematch_derived_features, prematch_derived_feature_names
from context import (build_context_features, context_feature_names,
                     stadium_table, unknown_stadiums)
from strength import (build_strength_features, strength_feature_names,
                      strength_history)
from build_features import build_features
# Trazabilidad: procedencia por columna y cobertura por partido
from lineage import (check_lineage, coverage_by_season, coverage_matrix,
                     feature_lineage, source_registry)
# Parte B (inferencia): reutiliza build_features por dentro
import build_scoring as BS
from build_scoring import build_scoring

PROC = ROOT/'data'/'02_processed'
master = pd.read_parquet(PROC/'master_matches.parquet')
master['date'] = master['date'].astype('string')   # ISO ordena bien; lo usan las dos partes
print('master:', master.shape, '| temporadas', master['season'].min(), '->', master['season'].max())
print('ultima fecha jugada:', master['date'].max())

# Parte A — Entrenamiento: `features_matches.parquet`

Construimos y auditamos la maquinaria de features sobre los partidos jugados. El resultado es la
tabla de entrenamiento **y** el código que la Parte B reutilizará tal cual.

## A.1 · Limpieza: qué se excluye y por qué

La "limpieza" clave no es tirar filas: es **separar** las columnas que describen el propio
partido (fuga) de las que se conocen antes. `columns.py` es la única fuente de verdad:

- **`PREMATCH_PASSTHROUGH`** → cuotas, Elo, valor de plantilla, fatiga, bajas y rotación del
  once. Pasan tal cual.
- **fuga** (`leakage_columns`) → goles, todas las `sofa_*`, xG/rating crudos. **Prohibidas en crudo**;
  solo materia prima de la forma.

In [ ]:
leak = C.leakage_columns(master.columns)
passth = [c for c in C.PREMATCH_PASSTHROUGH if c in master.columns]
print(f'columnas de FUGA (excluidas del set): {len(leak)}')
print(f'  ej.: {leak[:6]} ...')
print(f'columnas PRE-PARTIDO passthrough: {len(passth)}')
# El desglose se lee de las listas que declara `columns.py` y no de patrones sobre
# los nombres: un patron como "empieza por home_n_" reclamaria tanto el recuento de
# bajas como el numero de futbolistas utilizados, que son bloques distintos.
print(f'  mercado: {sum(c in C._MERCADO for c in passth)}',
      f'| elo: {sum(c in C._ELO for c in passth)}',
      f'| valor: {sum(c in C._SQUAD for c in passth)}',
      f'| fatiga: {sum(c in C._FATIGUE for c in passth)}',
      f'| bajas: {sum(c in C._ABSENCES for c in passth)}',
      f'| rotacion: {sum(c in C._LINEUPS for c in passth)}')


## A.2 · La primitiva de todo: el *team-match long*

Antes de crear ni una sola variable de forma hay que resolver un problema de **forma de los datos**. El
máster tiene una fila por partido con columnas `home_*` y `away_*`: es la vista natural para *ver un
partido*, pero es la vista **equivocada** para *medir a un equipo*. La forma de un equipo es la
secuencia temporal de **sus** partidos —da igual que jugara en casa o fuera—, y en el máster esa
secuencia está partida entre las columnas `home_` de unas filas y las `away_` de otras.

`to_team_long` deshace ese nudo: explota cada partido en **2 filas**, una por equipo, cada una desde
**su propia perspectiva**. Donde el máster decía `home_xg`/`away_xg`, el long dice simplemente `xg`
(el que generó *ese* equipo) y `xga` (el que le generaron). Lo mismo con `gf`/`ga`, `points`, `ppda`,
`rating`, tiros, córners y la forma de la distribución de remates (`shot_xg_mean`, `shot_dist_mean`,
`shot_setpiece_xg_pct`, `shot_xg_max`)... Además arrastra los escalares **pre-partido** que luego necesitan otras
familias (`elo`, `p_mkt_win`, `odds_win`), y marca `is_home` para poder reconstruir la vista por
partido al final.

Entre esos escalares pre-partido viaja uno que merece mención aparte: **`opp_elo`, el Elo del rival**.
Es lo que convierte una secuencia de resultados en una secuencia *interpretable*. Doce puntos en cinco
jornadas no significan lo mismo contra los cinco primeros que contra los cinco últimos, y sin el rival
en la fila esa distinción es invisible. Al viajar junto al resultado, la dificultad del calendario
admite exactamente el mismo tratamiento de media móvil que cualquier otro atributo (A.3.1), y permite
además contrastar cada resultado con **lo que el Elo esperaba de él antes de jugarse** — un contraste
que solo usa información conocida antes del pitido.

Ese reparto por perspectivas hace además que dos magnitudes **concedidas** aparezcan sin coste: la
calidad del remate que un equipo permite y la distancia a la que obliga a disparar
(`shot_xg_mean_against`, `shot_dist_mean_against`) no son una fuente nueva, sino el valor ofensivo
del rival leído desde la fila propia. Es la misma operación que ya produce `xga` a partir de `xg`, y
la razón de que el extractor de tiros no publique versiones defensivas: existirían dos veces.

Esta tabla es la **base común** sobre la que se apoyan casi todas las familias del catálogo que viene:
la forma reciente (`form.py`), las señales de mercado (`market_form.py`) y —con su propia variante
*dirigida por par de rivales*— el head-to-head (`h2h.py`). Clave única: `(match_key, team_key)`.

In [ ]:
long = to_team_long(master)
print('team-long:', long.shape, '| clave unica (match_key,team_key):',
      not long.duplicated(['match_key','team_key']).any())
demo = long[(long.team_key=='athletic_bilbao')&(long.season_year==2022)].sort_values('date')
demo[['date','opp_key','is_home','gf','ga','points','elo','opp_elo','xg','xga','rating']].head(8)

## A.3 · El catálogo de variables: una familia, un módulo

Esta es la parte central del notebook y, a la vez, el sitio para **consultar qué variables existen**.
La lógica de creación no está en un único bloque monolítico: cada **familia** de variables vive en su
propio módulo `.py`, con una única responsabilidad, y `build_features` (apartado A.5) las ensambla
todas por `match_key`. Los apartados A.3.1 a A.3.7 recorren esas familias una a una; cada uno explica
**qué** mide y **cómo** evita la fuga, y va seguido de una celda de código que **enumera en vivo las
columnas exactas** que crea (leídas de los propios módulos, así que el catálogo nunca se desincroniza
del código).

Las siete familias construidas, más el bloque que no se construye:

- **A.3.1 · Forma reciente y acumulada** (`form.py`) — el grueso: acumulados de temporada y la memoria
  reciente en sus **dos parametrizaciones** (una ventana corta de tres partidos y un decaimiento
  exponencial con semivida declarada), más la dureza del calendario recorrido, la forma por escenario,
  la efectividad y el momentum de Elo. Toda ella se calcula sobre el *team-long* de A.2.
- **A.3.2 · Mercado y psicología** (`market_form.py`) — rendimiento relativo a la cuota y arrastre
  emocional tras una victoria inesperada.
- **A.3.3 · Rivalidad histórica head-to-head** (`h2h.py`) — cómo le va a un equipo contra **ese** rival
  concreto, no contra el campo.
- **A.3.4 · Clasificación y trascendencia** (`standings.py`) — posición y presión en la tabla,
  reconstruida por *snapshot* a la fecha del partido, y lo que cada equipo se juega al entrar al partido.
- **A.3.5 · Derivadas pre-partido del mercado** (`prematch_derived.py`) — la estructura del precio
  (margen, desacuerdo entre casas, divergencia con la casa que ajusta con dinero informado) y la deriva
  de la línea entre apertura y cierre.
- **A.3.6 · Contexto del encuentro** (`context.py`) — calendario y geografía: franja de saque, fin de
  semana, desplazamiento del visitante y derbi. Es la única familia que no depende de cómo llegan
  los equipos, sino de dónde y cuándo se juega.
- **A.3.7 · Fuerza latente de ataque y defensa** (`strength.py`) — las dos componentes que el Elo funde
  en un escalar, estimadas con el modelo de goles, y la terna 1X2 que se deriva de ellas.
- **A.3.8 · El bloque *passthrough*** (`columns.py`) — lo que **no** se construye porque ya se conoce:
  precio de consenso, Elo, valor de plantilla, fatiga, bajas y rotación. Se cataloga aquí para que el
  inventario esté completo.

**Una convención común, y quién la decide.** Casi todas las familias calculan sus magnitudes por equipo
en el *long* y después vuelven a la vista de 1 fila/partido. Ahí se abre la pregunta de qué publicar:
`home_x`, `away_x` y `x_diff` son linealmente dependientes, de modo que emitir los tres regala una
columna redundante por atributo. La decisión no la toma cada módulo por su cuenta sino
`features/catalogo.py`, que declara para cada atributo si sale como **par**, como **diferencia** o
—solo en el núcleo de la forma— como ambos, y los módulos consultan esa declaración al escribir. El
criterio es cuál de las dos lecturas es la informativa: la diferencia cuando lo que importa es la
ventaja relativa (forma, historial, clasificación, trascendencia), y el par cuando el nivel absoluto
informa por sí mismo (Elo, valor de plantilla, bajas, fatiga, rotación: dos equipos con cuatro bajas
cada uno no producen el mismo partido que dos equipos sin ninguna, y la diferencia no distingue ambos
casos). Queda un cuarto caso, para lo que no se mide por equipo sino por **encuentro**: las derivadas
de A.3.5, el contexto de A.3.6, la terna de A.3.7 y la tasa de empate del h2h salen una sola vez y sin
rótulo de lado, porque preguntar «¿la del local o la del visitante?» no significa nada cuando el número
es el mismo para los dos.

**Y una consecuencia que conviene anticipar.** Como la mayoría de los atributos salen solo como
diferencia, la fila de partido ya no contiene el valor de cada equipo por separado. Eso no es una
pérdida de información para el modelo —la resta es la lectura que se ha decidido usar— pero sí
complicaría auditar los módulos, que es algo que este notebook hace en varios sitios. Por eso
`standings.py`, `h2h.py` y `strength.py` exponen además un accesor al frame por equipo
(`standings_history`, `h2h_history`, `strength_history`) del que las auditorías tiran cuando necesitan
mirar dentro del bloque.

### A.3.1 · Forma reciente y acumulada — `form.py`

Es la familia más grande y la que da nombre a "forma leak-safe". Toda ella nace del *team-long* de A.2,
agrupando por `(season_year, team_key)` (la tabla y la forma se **reinician cada temporada**) y con el
patrón anti-fuga en el centro: `shift(1)` para que la fila del partido *t* **nunca** vea su propio
resultado.

#### Los catorce atributos, y por qué son catorce

La fuente permite describir cómo llega un equipo con varias decenas de magnitudes, y construirlas todas
es la manera más rápida de acabar con una matriz de varios cientos de columnas en la que la mayoría son
la misma medida repetida. El catálogo fija en su lugar **un atributo por concepto medido**:

- **`points`, `gf`, `ga`** — el resultado y sus dos mitades. La diferencia de goles no se añade porque
  es exactamente `gf − ga`, y un estimador lineal la reconstruye sin ayuda.
- **`npxg`, `npxga`** — el peligro generado y concedido, sin penaltis. Entre `xg` y `npxg` se queda el
  segundo: es el primero sin el ruido de una jugada que no describe el juego del equipo.
- **`xpts`** — los puntos merecidos, que resumen las dos anteriores en la escala del objetivo.
- **`sot`, `big_chances`** — volumen de remate y ocasión clara: llegar, y llegar bien. Los remates
  totales y los córners miden lo mismo que el primero con más ruido.
- **`ppda`, `deep`** — presión y progresión, es decir el **estilo** y no el acierto.
- **`shot_xg_mean`, `shot_xg_mean_against`** — la calidad del remate medio, propio y concedido, que es
  lo que separa disparar mucho y mal de disparar poco y bien.
- **`rating`** — la nota agregada de SofaScore, como resumen de lo que las anteriores no capturan.
- **`opp_elo_z`** — la dureza del calendario recorrido, sin la cual todo lo anterior se lee fuera de
  contexto. El sufijo `_z` no es decorativo: el Elo viaja **tipificado dentro de la jornada** (ver
  notebook 02, §6), de modo que la magnitud son desviaciones respecto a la media de la liga y no
  puntos de rating.

Sobre esos catorce atributos se construyen los sub-bloques:

- **Acumulado de temporada** (`cum_points_pg`, `cum_xgd_pg`): media por partido de lo acumulado *antes*
  del encuentro, vía `cumsum().shift(1)` dividido por partidos previos. El acumulado de diferencia de
  goles se cae por ser la resta de dos acumulados que ya viajan; el de xG se queda porque es la versión
  de esa resta que no depende de la suerte en la definición.
- **Forma reciente a 3 partidos** (`form_<attr>_3`): media móvil `shift(1).rolling(3).mean()`.
- **Forma exponencial** (`form_<attr>_ewm`): los **mismos** atributos, con `shift(1).ewm(halflife=3)`.
- **Forma por escenario** (`venue_form_points_5`, `venue_form_gd_5`): rolling **solo** sobre los
  partidos en la misma condición (el local mirando sus partidos en casa, el visitante los suyos fuera).
- **Efectividad** (`effectiveness`): puntos acumulados / tiros a puerta acumulados (ambos con `shift`).
- **Momentum de Elo** (`elo_mom_5_z`): fuerza tipificada actual − la de hace 5 partidos, es decir
  cuántas desviaciones ha ganado o perdido el equipo respecto a la liga. El Elo es pre-partido y
  cruza el corte de temporada a propósito (no se reinicia en agosto). Que la variación se mida sobre
  la `z` y no sobre el nivel tiene una consecuencia concreta: un momentum calculado en puntos de
  rating leería el rebase de escala de ClubElo del 18-10-2022 como un salto de forma de los veinte
  clubes a la vez.
- **Estilo** (`playstyle`, categórica): etiqueta ofensivo / equilibrado / defensivo a partir de una
  media de diferencia de goles con memoria de la temporada previa. Esa media es un **insumo interno**
  y no sale a la matriz: lo que se publica es la etiqueta, que es la lectura que aporta algo por encima
  de los acumulados que ya viajan.

#### Dos parametrizaciones, no tres

"Forma reciente" no es una magnitud observable: es una **hipótesis sobre cómo se olvida**. La
ponderación exponencial es la memoria por defecto porque decae de forma continua a un ritmo fijado por
una **semivida** (`EWM_HALFLIFE = 3` partidos: a tres partidos de distancia un resultado pesa la mitad
que el último; a seis, la cuarta parte) y no obliga a elegir un corte arbitrario. Se le añade una
**ventana corta de tres partidos**, que afirma dos cosas distintas y menos evidentes —que el tercer
partido pesa exactamente lo mismo que el último, y que el cuarto no pesa nada—, porque «cómo llega
ahora mismo» y «cómo ha ido últimamente» son preguntas distintas y la respuesta a una no contiene la
otra.

Lo que **no** hay es una tercera ventana intermedia. Una ventana de cinco partidos no plantea una
tercera pregunta, solo una tercera respuesta a la que ya responden las dos anteriores, y su coste no es
de disco: cada atributo adicional entra en el espacio sobre el que después se selecciona, y una
selección sobre un espacio grande elige con alta probabilidad ruido que se parece a señal. El apartado
A.3.1-bis comprueba en dato que las dos parametrizaciones que sí se construyen no son una copia la una
de la otra.

La semivida se **declara, no se ajusta**. Optimizarla contra el resultado convertiría una descripción
del pasado en un hiperparámetro más, con su correspondiente riesgo de sobreajuste de selección.

#### La dureza del calendario

Una media de puntos es ciega al calendario: nueve puntos contra los tres colistas y nueve contra los
tres primeros producen exactamente el mismo `form_points_3`, y son cosas distintas. `opp_elo_z` —la fuerza relativa
del rival que A.2 dejó en la fila— promediada en las dos parametrizaciones mide literalmente el nivel
medio de los rivales a los que se ha enfrentado el equipo, y es lo que permite leer el resto de la
forma en contexto.

El **residuo de rendimiento** frente a la expectativa Elo (`elo_surprise = S − E`) sí se calculaba en
versiones anteriores y **ya no se construye**. La razón es que sus dos ingredientes viajan enteros y por
separado: `S` es el resultado, que es lo que miden `points` y `gf`/`ga`, y `E` es una función
determinista de la diferencia de Elo, que viaja en el par `home_elo_z`/`away_elo_z` junto a la dureza
del calendario. Una columna que es la resta de dos cosas que el modelo ya tiene no aporta información; solo
añade una medida más que imputar, escalar, podar y explicar. La función que calcula `E`
(`prematch_derived.elo_expected_score`) se conserva porque los bancos de prueba la ejercen, pero
ninguna columna de la matriz depende de ella.

#### Por qué la calidad de ocasión solo puede entrar por aquí

Las magnitudes de calidad de ocasión llegan al máster describiendo **el partido que se quiere
predecir**: cuánto valió el remate típico de ese encuentro. Usarlas tal cual sería pronosticar el
resultado a partir de las ocasiones que lo produjeron, de modo que `features/columns.py` las declara
como fuga y ninguna ruta las deja pasar al conjunto de variables. La media móvil de esta sección es su
**único** aprovechamiento legítimo, y lo es por el mismo mecanismo que hace legítimo usar el xG: el
`shift(1)` garantiza que la fila del partido *t* solo ve remates de partidos anteriores.

Conviene contrastarlo con el bloque de rotación, que procede de una fuente igual de reciente y sin
embargo **sí** pasa tal cual. La diferencia no está en la fuente sino en qué describe cada columna:
`home_xi_continuity` cuenta lo que el equipo hizo en su último partido —ya desplazado en el
extractor—, mientras que `home_shot_xg_mean` cuenta lo que ocurre en este. Misma procedencia, estatuto
opuesto.

De las seis magnitudes que el extractor de tiros publica, el catálogo se queda con **dos**: la calidad
del remate medio propio y la concedida. La distancia media al arco y el peso del balón parado son otra
manera de decir lo mismo que la primera —un remate lejano es un remate de xG bajo— y el xG máximo del
partido es un extremo de la distribución cuyo centro ya viaja. Que el bloque sea **opcional** aguas
arriba no cambia nada de lo anterior: si el volcado de tiros no se ha ejecutado, las dos columnas
llegan enteras a `NaN`, sus medias móviles nacen vacías y la poda por constancia del preprocesado las
retira antes de imputar nada.

Ninguna columna de esta familia lleva el token `close`, así que el bloque completo pertenece al
**régimen semanal** y sobrevive íntegro al filtro de `columns.weekly_unavailable()`.

In [ ]:
form = build_form_features(master)
import form as F
import catalogo as CAT
_names = form_feature_names(); _num, _cat = _names['numeric'], _names['categorical']
_W = F.WINDOWS[0]

def _b(pred): return [b for b in _num if pred(b)]
bloques = {
    'acumulado de temporada       (cumsum.shift / n)': _b(lambda b: b.startswith('cum_')),
    f'forma reciente N={_W}            (shift.rolling.mean)': _b(lambda b: b.startswith('form_') and b.endswith(f'_{_W}')),
    f'forma exponencial            (shift.ewm hl={F.EWM_HALFLIFE:g})': _b(lambda b: b.endswith('_ewm')),
    'forma por escenario (casa / fuera)': _b(lambda b: b.startswith('venue_form_')),
    'recorrido, efectividad y momentum de Elo': ['matches_prev', 'effectiveness', 'elo_mom_5_z'],
}
print(f'FORMA (form.py) -> {len(_num)} bases numericas + {len(_cat)} categorica\n')
for k, v in bloques.items():
    print(f'- {k}  [{len(v)}]')
    print(f'      {v}')
print(f'- categorica: {_cat}')

# Cuantas columnas produce cada base NO es una constante del modulo: lo decide el
# catalogo. El nucleo de la forma sale como par Y como diferencia porque en el el
# nivel absoluto discrimina; el resto sale solo como diferencia. Por eso el recuento
# se le PREGUNTA al catalogo en vez de multiplicar por tres.
_par = [b for b in _num if len(CAT.lados(b)) == 3]
_cols = [c for b in _num + _cat for c in CAT.nombres(b)]
print(f'\nnucleo emitido como par + diferencia [{len(_par)}]:\n      {_par}')
print(f'resto emitido solo como diferencia   [{len(_num) - len(_par)}]')
print(f'\ncolumnas que el catalogo declara para esta familia : {len(_cols)}')
print(f'columnas reales devueltas por build_form_features   : {form.shape[1] - 1}')
assert set(_cols) == set(form.columns) - {'match_key'}, 'modulo y catalogo desincronizados'

# Las dos parametrizaciones cubren exactamente el mismo conjunto de atributos: es la
# condicion para que la comparacion del apartado siguiente sea una comparacion.
_ewm = {b[len('form_'):-len('_ewm')] for b in _num if b.endswith('_ewm')}
_ven = {b[len('form_'):-len(f'_{_W}')] for b in _num
        if b.startswith('form_') and b.endswith(f'_{_W}')}
print(f'\natributos con las DOS parametrizaciones: {len(_ewm & _ven)} de {len(_ewm | _ven)}')

### A.3.1-bis · ¿Aportan algo las dos parametrizaciones? (análisis exploratorio)

Duplicar la memoria en dos parametrizaciones cuesta dimensionalidad, y la dimensionalidad se paga: cada
columna adicional entra en el espacio sobre el que después se selecciona, y una selección sobre un
espacio grande elige con alta probabilidad ruido que se parece a señal. Antes de arrastrar las dos
lecturas hasta el preprocesado conviene comprobar **empíricamente** que no son redundantes: si la forma
exponencial fuese una copia disfrazada de la ventana corta, lo honesto sería quedarse con una y no
dejar que la selección descarte la otra en silencio.

Se miran tres cosas: cuánto se parecen ventana y exponencial atributo a atributo, cuánta información
sobre el calendario recorrido añade `opp_elo` a un mismo nivel de forma, y cuánta señal cruda aporta
cada lectura sobre el objetivo 1X2 **medida solo en desarrollo** (la temporada de holdout no se toca ni
para mirar).

In [ ]:
# --- 1) ¿Es la exponencial una copia de la ventana corta? -------------------
# Todo lo que sigue se calcula SOLO sobre desarrollo: la temporada de holdout no
# se mira ni para un diagnostico descriptivo.
fm = form.merge(master[['match_key','season','ftr']], on='match_key', how='left')
fm_dev = fm[~fm['season'].isin(C.HOLDOUT_SEASONS)].copy()
print(f'partidos de desarrollo: {len(fm_dev)}  (holdout excluido: {C.HOLDOUT_SEASONS})')

# La comparacion se hace sobre las columnas de DIFERENCIA porque es el unico lado en
# que las dos parametrizaciones coexisten para TODOS los atributos: del nucleo se
# emite ademas el par, pero de la ventana corta nunca, asi que home_ contra home_ no
# seria una comparacion posible fuera de esos seis atributos.
filas = []
for a in CAT.FORMA_ATRIBUTOS:
    cw, ce = f'form_{a}_{_W}_diff', f'form_{a}_ewm_diff'
    if not {cw, ce} <= set(fm_dev.columns):
        continue
    sub = fm_dev[[cw, ce]].dropna()
    if len(sub) < 100:
        continue
    filas.append({'atributo': a, 'n': len(sub),
                  f'r(ewm, w{_W})': sub[ce].corr(sub[cw]),
                  f'sd ventana {_W}': sub[cw].std(), 'sd exponencial': sub[ce].std()})
cmp_par = pd.DataFrame(filas).set_index('atributo').round(3)
print('Correlacion entre las dos parametrizaciones de la MISMA memoria (solo desarrollo):')
display(cmp_par)

_r = cmp_par[f'r(ewm, w{_W})'].abs()
_umbral = 0.95
print(f'\nr medio {_r.mean():.3f} | minimo {_r.min():.3f} sobre {_r.idxmin()} | '
      f'maximo {_r.max():.3f} sobre {_r.idxmax()}')
print(f'atributos por encima del umbral de colinealidad ({_umbral}): '
      f'{list(cmp_par.index[_r >= _umbral]) or "ninguno"}')
print('Lectura: alta, como debe ser -miden lo mismo-, pero no identica: la ventana')
print('corta responde a "como llega ahora" y la exponencial a "como ha ido ultimamente".')
print('Los pares que superen el umbral no se arbitran aqui sino en la poda por')
print('colinealidad del notebook 04, que es donde esa decision se toma con dato.')

In [ ]:
# --- 2) ¿Que anade la dureza del calendario? --------------------------------
# Se agrupan los partidos por su DIFERENCIA de forma en puntos y se mira, dentro de
# cada grupo, cuanto varia la diferencia de nivel de los rivales ya enfrentados. Si
# el calendario no anadiera nada, seria constante dentro de cada grupo.
sub = fm_dev[['form_points_ewm_diff', 'form_opp_elo_z_ewm_diff']].dropna()
sub = sub.assign(bucket=sub['form_points_ewm_diff'].round(1))
g = (sub.groupby('bucket')
        .agg(n=('form_opp_elo_z_ewm_diff', 'size'),
             rival_p10=('form_opp_elo_z_ewm_diff', lambda s: s.quantile(.10)),
             rival_mediana=('form_opp_elo_z_ewm_diff', 'median'),
             rival_p90=('form_opp_elo_z_ewm_diff', lambda s: s.quantile(.90)))
        .query('n >= 40').round(2))
print('Para una MISMA diferencia de forma en puntos, ¿que rivales habia detras?')
display(g.head(12))
_ancho = (g['rival_p90'] - g['rival_p10']).median()
print(f'\nRango intercuantil (p10-p90) tipico de la diferencia de Elo medio del rival')
print(f'dentro de un mismo nivel de forma en puntos: {_ancho:.0f} puntos Elo. Esa es')
print('exactamente la informacion que la media de puntos NO puede ver y que opp_elo')
print('recupera: nueve puntos contra los tres colistas y nueve contra los tres')
print('primeros producen el mismo form_points, y no son lo mismo.')

In [ ]:
# --- 3) Senal cruda de cada lectura sobre el objetivo 1X2 (solo desarrollo) --
from sklearn.feature_selection import mutual_info_classif

y = fm_dev['ftr'].map({'H':0,'D':1,'A':2})

cands = []
for a in ['points', 'gf', 'ga', 'npxg', 'xpts', 'rating', 'opp_elo']:
    cands += [f'form_{a}_{_W}_diff', f'form_{a}_ewm_diff']
cands += ['cum_points_pg_diff', 'cum_xgd_pg_diff']
cands = [c for c in cands if c in fm_dev.columns]

X = fm_dev[cands]
msk = X.notna().all(axis=1) & y.notna()
mi = mutual_info_classif(X[msk], y[msk].astype('int'), random_state=0)
mi_tab = (pd.DataFrame({'feature': cands, 'MI': mi,
                        'cobertura': X.notna().mean().values})
          .assign(MIxcob=lambda d: d.MI * d.cobertura)
          .sort_values('MIxcob', ascending=False).round(4).reset_index(drop=True))
print(f'Informacion mutua con el objetivo 1X2 (n={int(msk.sum())} partidos de desarrollo):')
display(mi_tab)
print('\nNo se decide nada aqui: es un diagnostico univariante, y la seleccion real es')
print('la busqueda hacia delante del notebook 05, que es multivariante y se ejecuta')
print('sobre un bloque de temporadas separado del que despues mide. Lo que se comprueba')
print('es que ninguna de las dos parametrizaciones es informativamente vacia.')

**Qué se concluye.** Las tres comprobaciones apuntan en la misma dirección: la exponencial correlaciona
alto con la ventana corta —como debe, miden lo mismo— pero no coincide con ella, así que no es una
copia; y para un mismo nivel de forma en puntos, el Elo medio de los rivales varía en decenas de
puntos, información que la media de puntos por sí sola no puede contener. Las dos parametrizaciones se
arrastran, por tanto, hasta la etapa de selección, que es el único sitio del pipeline donde una
decisión de este tipo puede tomarse **con evidencia y sin mirar el holdout**.

Conviene subrayar el estatus de este apartado: es **exploratorio y descriptivo**, no decisorio. La
información mutua univariante ignora la redundancia entre candidatas —dos columnas idénticas
puntuarían idéntico y alto— y por eso no puede sustituir a la selección hacia delante del notebook 05,
que evalúa cada candidata condicionada a las ya elegidas. Su papel aquí es descartar la hipótesis nula
fuerte ("la segunda parametrización no aporta nada"), no ordenar las variables.

Y conviene subrayar también qué **no** se está comprobando. Este apartado defiende las dos lecturas que
el catálogo construye; no defiende las que no construye, porque de esas no hay columna que mirar. La
ventana intermedia de cinco partidos, los residuos de xG o el residuo de sorpresa frente al Elo no se
descartan aquí con un diagnóstico: se descartan antes, en la declaración, y la razón es que ninguno
plantea una pregunta que las columnas declaradas no respondan ya.

### A.3.1-ter · ¿Dice la calidad de ocasión algo que el volumen y el xG no digan?

La familia de forma ya contenía dos lecturas del ataque: **cuánto** remata un equipo (`form_sot_*`,
`form_big_chances_*`) y **cuánto peligro** acumula en total (`form_npxg_*`). Las dos magnitudes de
calidad de ocasión —el xG del remate medio, propio y concedido— añaden una tercera: **cómo se reparte**
ese peligro entre los disparos. Esa afirmación merece comprobarse antes de arrastrar cuatro columnas
más hasta el preprocesado.

La hipótesis nula es incómoda y hay que tomarla en serio: si el xG medio por remate fuese esencialmente
`npxg / sot`, sería una función determinista de dos atributos que ya existen y no aportaría
información, solo dimensionalidad. Lo que sigue mide exactamente eso, **solo sobre desarrollo**, con el
mismo criterio del apartado anterior: la temporada de holdout no se mira ni para un diagnóstico
descriptivo.

In [ ]:
# --- ¿Es la calidad de ocasion una funcion del volumen y el xG ya presentes? --
_CAL = ['shot_xg_mean', 'shot_xg_mean_against']
_cols_cal = [c for a in _CAL for c in (f'form_{a}_{_W}_diff', f'form_{a}_ewm_diff')
             if c in fm_dev.columns]
_vivo = [c for c in _cols_cal if fm_dev[c].notna().any()]

if not _vivo:
    print("El bloque de calidad de ocasion llega vacio: falta el volcado de tiros\n"
          "(`python scripts/extractors/understat_shots.py backfill && ... process`).\n"
          "Las medias moviles existen pero no tienen materia prima, y la poda por\n"
          "constancia del notebook 04 las retirara. El resto del catalogo no cambia.")
else:
    _ref = [c for c in ['form_sot_ewm_diff', 'form_big_chances_ewm_diff',
                        'form_npxg_ewm_diff', 'form_npxga_ewm_diff']
            if c in fm_dev.columns]
    corr = fm_dev[_vivo + _ref].dropna().corr().loc[_vivo, _ref].round(3)
    print('Correlacion de la forma de CALIDAD contra la de VOLUMEN y peligro total:')
    display(corr.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1)
            .format('{:.2f}'))
    print(f'\ncorrelacion absoluta maxima observada: {corr.abs().to_numpy().max():.2f}')

    # La prueba directa de la hipotesis nula, y se hace sobre el ATRIBUTO en el
    # team-long y no sobre su media movil: si la media por remate fuera el cociente
    # de dos agregados que ya viajan, lo seria partido a partido, antes de que
    # ninguna ventana suavice la comparacion.
    _lg = long.merge(master[['match_key', 'season']], on='match_key', how='left')
    _lg = _lg[~_lg['season'].isin(C.HOLDOUT_SEASONS)]
    if {'shot_xg_mean', 'npxg', 'sot'} <= set(_lg.columns):
        s = _lg[['shot_xg_mean', 'npxg', 'sot']].dropna()
        s = s[s['sot'] > 0]
        cociente = s['npxg'] / s['sot']
        print(f"\nr(xG medio por remate, npxG / tiros a puerta) = "
              f"{s['shot_xg_mean'].corr(cociente):.3f}   (n={len(s):,})")
        print('No es 1 y no puede serlo: el cociente reparte el peligro entre los tiros')
        print('que fueron A PUERTA, mientras que la media se calcula sobre TODOS los')
        print('remates. Un equipo que dispara mucho desde lejos y acierta poco la')
        print('porteria sale alto en el cociente y bajo en la media, y esa discrepancia')
        print('es precisamente lo que la calidad de ocasion mide.')

**Qué se concluye.** La calidad de ocasión no es una reescritura del volumen: sus correlaciones con los
agregados que ya estaban son moderadas, y el xG medio por remate no coincide con el cociente
`npxG / tiros a puerta` porque no se calcula sobre el mismo conjunto de disparos. La magnitud concedida
merece mención aparte, porque era la que corría más riesgo de duplicar a `npxga`: el peligro concedido
dice **cuánto** permite un equipo, y esta dice **de qué tipo** es —pocas ocasiones claras o muchas
inocuas—, que es justo la distinción que un total no puede contener.

De las seis magnitudes que el extractor publica se conservan estas dos, y la elección tiene el mismo
criterio que todo el catálogo: la distancia media al arco es otra forma de decir lo que dice el xG por
remate —un disparo lejano es un disparo de xG bajo—, el peso del balón parado es un desglose de la
misma cantidad y el xG máximo del encuentro es un extremo de la distribución cuyo centro ya viaja.
Construirlas para que la poda las retirase después habría significado pagar su extracción, su
consolidación, su imputación y su escalado a cambio de nada.

Como en el apartado anterior, aquí no se decide nada: la comprobación descarta la hipótesis nula fuerte
y deja que sea la selección hacia delante del notebook 05, ajustada solo sobre el bloque de selección,
quien determine si estas bases sobreviven.

### A.3.2 · Mercado y psicología — `market_form.py`

También se apoya en el *team-long*, pero con una diferencia deliberada: aquí la memoria **cruza el corte
de temporada** (media exponencial EWM con vida media 6), porque ni el rendimiento relativo al mercado ni
los arrastres emocionales se reinician en agosto. Dos variables, ambas con `shift(1)`:

- **`relative_perf`**: victorias reales (EWM de `won`) / victorias esperadas por la cuota (EWM de
  `p_mkt_win`). Por encima de 1 el equipo rinde **mejor** de lo que le da el mercado; por debajo, peor.
  Es una señal de reversión a la media respecto a la "opinión agregada" de las casas. Sale como
  **diferencia**: lo que informa es cuál de los dos viene rindiendo por encima de su precio.
- **`post_big_win`**: 1 si el partido **anterior** fue una victoria inesperada (cuota de victoria > 4.0,
  ~menos del 25% implícito). Hipótesis: posible relajación posterior. Sale como **par**, porque es un
  estado de cada equipo y restar dos indicadores binarios pierde el caso en que los dos vienen de una
  sorpresa.

La resaca del favorito que tropieza (`post_big_fav_loss`) se calculaba en versiones anteriores y **no se
construye**: su información mutua con el resultado es del orden del ruido de muestreo, de modo que su
único efecto real era añadir una columna al espacio sobre el que después se selecciona. Un indicador que
se activa en un puñado de partidos por temporada y no separa el objetivo es exactamente el tipo de
columna que una búsqueda greedy puede elegir por azar.

In [ ]:
market = build_market_form_features(master)
cols = [c for c in market.columns if c != 'match_key']
print(f'MERCADO / PSICOLOGIA (market_form.py) -> {len(cols)} columnas\n')
for base in market_form_feature_names():
    print(f'- {base:18s} : {[c for c in cols if base in c]}')

### A.3.3 · Rivalidad histórica head-to-head — `h2h.py`

Mide cómo le ha ido a un equipo contra **un rival concreto**, no contra el campo. Usa una variante
propia del *long*: uno **dirigido por par** `(equipo, rival)`, ordenado por fecha, con tasas de
decaimiento exponencial (EWM, más peso a los duelos recientes) y `shift(1)` para no ver el actual. La
memoria cruza temporadas a propósito: la historia de un clásico no se borra cada agosto. Sin
enfrentamientos previos las tasas se rellenan **neutras (1/3)** y la diferencia de goles h2h queda
`NaN` (se imputa en el 04).

**Tres columnas, y las tres decisiones que las dejan en tres.**

- **`h2h_win_rate_ewm_diff`** — tasa histórica ponderada de victoria contra ese rival, como diferencia
  entre los dos lados. La tasa de **derrota** no se construye: es uno menos las otras dos, de modo que
  un estimador lineal la reconstruye sin ayuda.
- **`h2h_gd_ewm_diff`** — diferencia de goles media ponderada en el histórico del duelo.
- **`h2h_draw_rate_ewm`** — tasa histórica ponderada de empate, y esta sale **sin lado**. La razón es
  que un empate lo es para los dos equipos a la vez: ambas filas dirigidas llevan exactamente el mismo
  número, su resta es cero en todas las filas del histórico y emitirla como diferencia añadiría una
  columna constante. Es una magnitud del **enfrentamiento**, no del equipo.

La **ventana rectangular de ocho enfrentamientos** que existía en versiones anteriores tampoco se
construye. Responde a la misma pregunta que la exponencial con otro supuesto de olvido, y en un
historial donde la muestra es de uno o dos partidos por temporada las dos apenas se distinguen: la
ventana de ocho cubre unos cuatro años de rivalidad, que es más o menos el horizonte que la semivida de
seis enfrentamientos ya pondera.

**El frame por equipo sigue disponible.** Como de las dos magnitudes que miden al equipo solo sale la
diferencia, y de una resta no se recupera el historial de ninguno de los dos, `h2h.py` expone
`h2h_history(master)`: el long dirigido con las dos filas de cada partido. Es donde la media
exponencial se calcula y donde la identidad que da sentido al bloque —que las dos filas describen el
mismo historial visto desde lados opuestos— es siquiera formulable. El apartado A.7 la comprueba
sobre datos reales tirando de ese accesor.

In [ ]:
h2h = build_h2h_features(master)
cols = [c for c in h2h.columns if c != 'match_key']
print(f'HEAD-TO-HEAD (h2h.py) -> {len(cols)} columnas\n')
for base in h2h_feature_names():
    # Cada atributo sale en los lados que declara el catalogo, y solo en esos: los
    # dos que miden al EQUIPO viajan como diferencia, el que mide al PAR una sola vez
    # y sin rotulo de lado.
    print(f'- {base:16s} {str(CAT.lados(base)):22s} : {CAT.nombres(base, "h2h_")}')
assert set(cols) == {c for b in h2h_feature_names() for c in CAT.nombres(b, 'h2h_')}


### A.3.4 · Clasificación y trascendencia — `standings.py`

Reconstruye la tabla tal como estaba **antes** de cada partido y de ahí saca la presión contextual.
Método = **snapshot por fecha**: para la fecha *D* de un partido se suman puntos/gd/gf de cada equipo en
sus partidos jugados con fecha **estrictamente anterior** a *D* y se ordena la tabla de 20. Esto no solo
es leak-safe, sino que además es lo correcto en la ruta de inferencia: un partido futuro se clasifica
contra la tabla **real de los 20 equipos** a esa fecha, no contra el subconjunto que se está prediciendo
(que es el error en el que caería un enfoque por "número de jornada" sobre un máster extendido parcial).

**Dónde está cada equipo.**

- **`prev_position`**: puesto (1 = líder) al entrar al partido.
- **`position_zone`** (categórica): `champions` (≤4) / `europa` (5-6) / `mid_table` (7-17) /
  `descenso` (≥18). Es la única de este bloque que sale como **par**: es ordinal y su resta no
  significa nada.
- **`points_above_drop`**: puntos de colchón por encima del primer puesto de descenso (18º).
- **`dynamic_pos_change`**: posición final de la temporada previa − posición actual (>0 = va mejor de
  como acabó el año pasado; <0 = peor).

**Qué se juega.** La posición dice dónde está un equipo, pero no qué tiene delante. Dos equipos
decimoterceros no llegan igual al partido si uno pelea por Europa a tres puntos y el otro no puede ni
descender ni alcanzarla: el segundo compite por el orgullo. Este bloque mide esa distancia a cada
objetivo y, sobre ella, decide si el objetivo sigue vivo.

- **`points_to_leader` / `points_to_champions` / `points_to_europa`**: puntos que separan al equipo del
  1.º, del 4.º y del 6.º de la tabla previa. Un valor ≤ 0 significa que el equipo ya ocupa ese puesto o
  uno mejor, de modo que la variable es continua y con signo, no un indicador de zona.
- **`matches_left`**: partidos que le restan (38 menos los disputados). Es lo que fija el **techo**
  de cada equipo —sus puntos actuales más tres por partido pendiente— y, con él, si una distancia
  sigue siendo salvable. Sale **solo del local**, porque a mediados de temporada los dos equipos de un
  partido llevan casi siempre el mismo número de jornadas disputadas y la segunda columna repetiría la
  primera.
- **`safe_from_drop`**: 1 cuando la permanencia está matemáticamente asegurada.

**La regla de decisión es una cota conservadora, y se resuelve contando equipos.** Un objetivo se
declara cerrado solo si lo está *con certeza*. Para cada equipo de la tabla previa se calcula su
**techo** —los puntos que tendría ganando todo lo que le resta— y sobre esos techos se cuenta: un equipo
está **salvado** cuando hay ya tantos rivales con el techo por debajo de sus puntos actuales como plazas
de descenso hay en juego, tres, porque entonces terminará por delante de todos ellos sume lo que sume a
partir de ahora; y **Europa le es inalcanzable** cuando hay seis rivales cuyos puntos actuales superan su
propio techo, porque los seis acabarán por delante.

Contar equipos, en lugar de medir la distancia contra el corte del momento, es lo que mantiene la cota
válida cuando el calendario va desigual. Un rival con un partido aplazado puede sumar más puntos de los
que restan al equipo que se evalúa, de modo que una regla apoyada en los partidos pendientes *propios*
declararía salvaciones que el calendario todavía no garantiza; aquí cada rival se acota con los partidos
que le quedan a él. Las dos comparaciones son además **estrictas**: un rival que llega justo a los puntos
del equipo no cuenta como descartado, porque el desempate podría no acompañar. El diseño acepta así
declarar *tarde* —etiquetar de "en juego" una situación que ya estaba resuelta— y nunca declarar *de
más*, que es el error que ensuciaría la variable. Por la misma razón las banderas quedan **vacías**, y no
a cero, mientras no hay tabla previa de la que contar.

**Qué de todo eso sale a la matriz, y qué se queda dentro.** El indicador de Europa inalcanzable
(`europe_unreachable`) se calcula —hace falta para lo que viene después— pero **no se emite**: es el
complemento casi determinista de `points_to_europa`, que ya viaja como distancia continua y con signo.
Y el indicador de partido intrascendente tampoco sale por equipo, sino una sola vez y como producto:

- **`stakes_none_match`**: los **dos** equipos sin nada en juego a la vez, que es la situación de la que
  el dominio habla cuando dice que un partido "no vale para nada". No es una combinación lineal de los
  dos indicadores por lado ni de su diferencia, así que un estimador aditivo no puede reconstruirla y
  merece columna propia; por lado, en cambio, no aporta nada que las distancias a cada objetivo no
  digan ya.

En total el módulo emite **once columnas**: siete diferencias, la zona de tabla por lado, los partidos
que restan al local y el indicador de partido intrascendente. La aritmética por equipo —el colchón sobre
el descenso, el techo alcanzable, las dos banderas— vive en `standings_history(master)`, que se expone
precisamente para que las auditorías de las celdas siguientes puedan mirarla: de una resta no se
recupera el valor de ninguno de los dos equipos.

In [ ]:
standings = build_standings_features(master)
cols = [c for c in standings.columns if c != 'match_key']
_sn = standings_feature_names()
print(f'CLASIFICACION (standings.py) -> {len(cols)} columnas\n')
for base in _sn['numeric'] + _sn['categorical']:
    tipo = 'num' if base in _sn['numeric'] else 'cat'
    print(f'- [{tipo}] {base:20s} : {[c for c in cols if base in c]}')

### A.3.4-bis · ¿Se sostiene la cota? (auditoría descriptiva)

Las dos banderas del bloque anterior son afirmaciones fuertes: dicen que un objetivo está **decidido**.
Una regla así solo es defendible si nunca se contradice con lo que acabó pasando, y eso puede
comprobarse sobre el propio histórico sin necesidad de modelo alguno: basta con enfrentar cada
declaración a la tabla final de esa temporada. Si un solo equipo declarado matemáticamente salvado
hubiese descendido, la cota sería falsa y la variable, ruido con apariencia de certeza.

Se miran tres cosas. Primero, con qué frecuencia y en qué momento del calendario deja de haber algo en
juego, que es lo que dice si la variable tiene recorrido o es casi siempre cero. Segundo, la
verificación de la cota contra el desenlace real. Tercero, cuánta temporada queda cuando cada objetivo
se cierra, que cuantifica el precio de haber elegido una regla conservadora. Todo ello **solo sobre
desarrollo**: la temporada de holdout no se mira ni para un diagnóstico descriptivo.

In [ ]:
# Todo lo que sigue se calcula SOLO sobre desarrollo. La aritmetica POR EQUIPO se
# consulta en standings_history, que es el frame donde vive: la fila de partido emite
# la mayoria de estos atributos como diferencia, y de una resta no se recupera el
# valor de ninguno de los dos lados.
from standings import standings_history

sth = (standings_history(master)
       .merge(master[['match_key', 'season']], on='match_key', how='left'))
sth_dev = sth[~sth['season'].isin(C.HOLDOUT_SEASONS)].copy()

st = standings.merge(master[['match_key', 'season', 'season_year', 'date']],
                     on='match_key', how='left')
st_dev = st[~st['season'].isin(C.HOLDOUT_SEASONS)].copy()
print(f'partidos de desarrollo: {len(st_dev)}  (holdout excluido: {C.HOLDOUT_SEASONS})')

# --- 1) ¿Cuando deja de haber algo en juego? --------------------------------
_un = ((sth_dev.groupby('match_key')['stakes_none'].max() > 0)
       .astype(float).rename('un_equipo').reset_index())
st_dev = st_dev.merge(_un, on='match_key', how='left')
st_dev['mes'] = pd.to_datetime(st_dev['date']).dt.month
st_dev['un_equipo'] = st_dev['un_equipo'].fillna(0.0)
st_dev['los_dos'] = st_dev['stakes_none_match'].fillna(0)
_orden = [8, 9, 10, 11, 12, 1, 2, 3, 4, 5, 6, 7]
frec = (st_dev.groupby('mes')
        .agg(partidos=('match_key', 'size'),
             pct_un_equipo=('un_equipo', 'mean'),
             pct_los_dos=('los_dos', 'mean'))
        .reindex([m for m in _orden if m in set(st_dev['mes'])]).round(3))
print('\nPartidos con al menos un equipo sin nada en juego, por mes:')
display(frec)

In [ ]:
# --- 2) La cota contra el desenlace real ------------------------------------
# Tabla FINAL de cada temporada, reconstruida desde el team-match long.
fin = (long[long['points'].notna()]
       .groupby(['season_year', 'team_key'], as_index=False)
       .agg(pts=('points', 'sum'), gd=('gd', 'sum'), gf=('gf', 'sum'))
       .sort_values(['season_year', 'pts', 'gd', 'gf'],
                    ascending=[True, False, False, False]))
fin['pos_final'] = fin.groupby('season_year').cumcount() + 1

# Las dos banderas y los partidos que restan se leen del frame por equipo: en la
# matriz de partido `safe_from_drop` solo sobrevive como diferencia, `matches_left`
# solo del local y `europe_unreachable` no sobrevive en absoluto.
aud = (sth_dev[['match_key', 'season_year', 'team_key', 'safe_from_drop',
                'europe_unreachable', 'matches_left']]
       .rename(columns={'safe_from_drop': 'salvado',
                        'europe_unreachable': 'sin_europa',
                        'matches_left': 'restan'})
       .merge(fin[['season_year', 'team_key', 'pos_final']],
              on=['season_year', 'team_key'], how='left'))

n_sal = int((aud['salvado'] == 1).sum())
mal_sal = int(((aud['salvado'] == 1) & (aud['pos_final'] >= 18)).sum())
n_eur = int((aud['sin_europa'] == 1).sum())
mal_eur = int(((aud['sin_europa'] == 1) & (aud['pos_final'] <= 6)).sum())
print(f'declaraciones de salvacion matematica : {n_sal:5d}  ->  descendieron       : {mal_sal}')
print(f'declaraciones de Europa inalcanzable  : {n_eur:5d}  ->  acabaron en Europa : {mal_eur}')
assert mal_sal == 0 and mal_eur == 0, 'la cota conservadora ha declarado de mas'

# --- 3) El precio de ser conservador ----------------------------------------
print('\nJornadas que restan cuando el objetivo ya esta cerrado:')
for f, etq in [('salvado', 'salvacion matematica'), ('sin_europa', 'Europa inalcanzable')]:
    s = aud.loc[aud[f] == 1, 'restan'].dropna()
    if len(s):
        print(f'  {etq:22s} n={len(s):5d}  mediana={s.median():.0f}  '
              f'maximo={s.max():.0f}')
    else:
        print(f'  {etq:22s} sin casos en desarrollo')

**Qué se concluye.** Ninguna declaración se contradice con la tabla final: entre todos los partidos de
desarrollo no hay un solo equipo al que la regla diera por salvado y terminara descendiendo, ni uno al
que declarara fuera de Europa y acabara en ella. Es la propiedad que se pedía, y es consecuencia
directa de haber elegido una cota —cuántos rivales no pueden ya cruzarse con el equipo— en lugar
de un pronóstico. La comprobación es además una **aserción**, no un texto: si una versión futura del
módulo relajara la cota, la celda rompería en lugar de imprimir un número peor.

El precio de esa elección se lee en el tercer bloque: los objetivos se cierran con pocas jornadas por
delante, de modo que la variable marca el tramo final del campeonato y no reparte ceros por toda la
temporada. Eso es exactamente lo que se quería medir —la trascendencia es un fenómeno de mayo— y
también la razón de que las distancias a cada corte se conserven aparte: son continuas, existen desde
la primera jornada y capturan la presión gradual que las banderas, por construcción, solo reconocen
cuando ya es total. Es también la razón de que de las dos banderas solo una llegue a la matriz por
equipo: `europe_unreachable` no dice casi nada que `points_to_europa` no diga antes y mejor.

Nótese de dónde ha salido el dato de esta auditoría. Las dos banderas y los partidos restantes se han
leído de `standings_history`, no del conjunto de variables, porque en él `safe_from_drop` solo
sobrevive como diferencia entre los dos equipos y `europe_unreachable` no sobrevive en absoluto. Ese es
el motivo de que el módulo exponga el frame por equipo: recortar el conjunto de variables no debe
implicar renunciar a poder comprobar lo que se recorta.

### A.3.5 · Derivadas pre-partido del mercado — `prematch_derived.py`

A diferencia de las anteriores, aquí **no hay ventanas temporales ni `shift`**: cada variable se calcula
a partir de escalares que ya existen en el máster antes del pitido (cuotas de apertura de varias casas,
cuota de Pinnacle y cuota de cierre). No describen el resultado, sino el **estado** del mercado al
entrar al partido. Son *match-level*: una sola columna por partido, sin lado `home_`/`away_`.

**Estructura del precio (5 columnas).** La probabilidad implícita de consenso `mkt_p_*` llega ya
**normalizada a suma 1**, y esa normalización destruye dos señales que el precio en bruto sí contiene:
cuánto margen cobra el libro y cuánto discrepan las casas entre sí. Ninguna de las dos es redundante con
la probabilidad resultante, porque ambas desaparecen al dividir por la suma. Este bloque las recupera.

- **`mkt_overround_open`**: suma de las inversas de las tres cuotas de apertura **antes** de normalizar.
  Un valor de 1.00 sería un precio justo; el exceso sobre 1 es el margen. Un margen ancho señala un
  partido que el libro considera incierto o poco líquido: es una medida de la *confianza del mercado*
  que `mkt_p_*` ya no puede contener.
- **`mkt_spread_mean`**: desacuerdo entre casas, medido como la ventaja relativa del mejor precio
  disponible sobre el consenso, `(max − avg) / avg`, promediada sobre los tres signos. Cuando las casas
  coinciden el valor es casi cero; cuando discrepan, el partido está mal definido para el propio
  mercado.
- **`sharp_div_home` / `_draw` / `_away`**: divergencia entre Pinnacle y el consenso,
  `p_ps − p_consenso`, lado a lado. Pinnacle es una casa de margen estrecho que admite apostantes
  informados en lugar de limitarlos, por lo que su cotización suele estar mejor calibrada que la media
  del sector; un valor grande indica que el precio informado y la media del sector no cuentan la misma
  historia sobre el partido.

**Qué se calcula y no se emite, y por qué.** La probabilidad des-margen de **Pinnacle**
(`mkt_p_ps_*`) es el insumo de la divergencia anterior, pero no viaja como variable: correlaciona por
encima de 0.95 con el consenso —son dos cotizaciones del mismo partido— y meter las dos ternas
obligaría a la poda por colinealidad a romper el empate de forma arbitraria. Lo que Pinnacle aporta por
encima del consenso es precisamente en qué se **separa** de él, y eso es lo que se emite. Por la misma
razón desaparecen el desglose del *spread* por signo —el resumen escalar ya lo lleva—, el margen de
Pinnacle y el resumen de divergencia total, que es la media de los tres valores absolutos que ya viajan
enteros.

**Deriva de la línea (1 columna).**

- **`close_move_fav`**: desplazamiento de la probabilidad implícita (ya des-margen) entre la cuota de
  **apertura** y la de **cierre**, expresado siempre a favor del **favorito de apertura**. Captura hacia
  dónde se movió el dinero: el cierre suele ser el precio más eficiente. El movimiento por signo
  (`close_move_home/draw/away`) no se emite porque las tres columnas suman cero por construcción y este
  resumen es su lectura con signo estable.

**Los dos bloques no pertenecen al mismo régimen de información.** La estructura del precio se calcula
con cuotas de **apertura**, que existen días antes del partido, de modo que sus cinco columnas están
disponibles en el momento de emitir la predicción. La deriva de la línea necesita el **cierre**, que
solo existe minutos antes del saque. Por eso lleva el token `close` en el nombre:
`columns.weekly_unavailable` la aparta del régimen semanal por convenio de nombres, sin lista paralela
que mantener. La celda siguiente comprueba esa partición en vivo.

**Y una terna que se construye aquí sin ser una variable.** `mkt_p_b365_home/draw/away` es la
probabilidad implícita de Bet365 sin margen, y no entra al conjunto que ve el estimador —el consenso ya
lleva ese precio— sino que viaja como columna de **servicio**: es el patrón de comparación contra el que
se mide todo el trabajo, y tiene que estar sobre exactamente las mismas filas que el pronóstico para que
la comparación sea posible. `columns.SERVICIO_DERIVADO` es quien la mantiene fuera del conjunto de
variables.

**Sobre las guardas.** Una cuota corrupta no debe producir un número plausible. Antes de invertir se
descarta toda cuota decimal por debajo de 1.01, y una terna cuyo *overround* caiga fuera de [0.95, 1.60]
se anula entera (no es un precio 1X2 creíble); un `spread` negativo por redondeo se absorbe a cero y uno
superior al 100 % se anula. El resultado es que el hueco es explícito y lo resuelve la imputación por
causa del preprocesado, en lugar de entrar al modelo disfrazado de dato.

In [ ]:
derived = build_prematch_derived_features(master)
cols = [c for c in derived.columns if c != 'match_key']
import prematch_derived as PD
print(f'DERIVADAS PRE-PARTIDO (prematch_derived.py) -> {len(cols)} columnas '
      f'(match-level, sin home_/away_)\n')

bloques = {
    'estructura del precio (margen / desacuerdo / divergencia)': PD.PRICE_STRUCTURE,
    'deriva de la linea apertura -> cierre': PD.LINE_MOVEMENT,
    'patron de comparacion Bet365 (SERVICIO, no es variable)': PD.SERVICIO_B365,
}
for k, v in bloques.items():
    print(f'- {k}  [{len(v)}]')
    for base in v:
        s = derived[base].dropna()
        rango = (f'rango ({s.min():.3f}, {s.max():.3f})  media {s.mean():.3f}'
                 if len(s) else 'sin valores')
        print(f'      {base:20s} cobertura {derived[base].notna().mean():6.1%}   {rango}')

# El regimen NO se declara aqui: se deduce del nombre. Las columnas que dependen del
# cierre llevan el token 'close', asi que weekly_unavailable las aparta del semanal
# sola. Esta comprobacion es la que garantiza que el convenio se ha respetado.
fuera = set(C.weekly_unavailable(cols))
print(f"\nno conocibles al emitir la prediccion (token '{C._CLOSING_TOKEN}'): {sorted(fuera)}")
print('coinciden exactamente con la deriva de linea:', fuera == set(PD.LINE_MOVEMENT))
print('la estructura del precio queda integra en el semanal:',
      not (fuera & set(PD.PRICE_STRUCTURE)))

# Y la terna de Bet365 se construye pero NO es una variable: viaja como servicio.
print('\nBet365 declarado como servicio y fuera del set:',
      set(PD.SERVICIO_B365) <= set(C.SERVICIO))

### A.3.6 · Contexto del encuentro: calendario y geografía — `context.py`

Las cinco familias anteriores describen **cómo llegan** los equipos. Esta describe **dónde y cuándo** se
juega, que es el otro condicionante clásico del partido y el único que no depende de nadie: el
calendario lo publica la competición con semanas de antelación y la geografía no cambia. Son, por
tanto, pre-partido en el sentido fuerte —existen antes del pitido inicial y también en el momento de
emitir la predicción—, así que no necesitan `shift` ni pertenecen al bloque de cierre. Como el bloque
de derivadas, son variables **de partido**: una sola columna, sin lados `home_`/`away_`.

**Calendario (2 variables, desde `kickoff_ts`).**

- **`kickoff_slot`** (categórica ordinal): franja de emisión, `mediodia` (< 15 h) < `tarde` (15-19 h) <
  `noche` (≥ 19 h). El sello viaja en UTC y se convierte a `Europe/Madrid` antes de recortar: leerlo en
  UTC desplazaría a la víspera todos los saques nocturnos de invierno y contaminaría dos franjas a la vez.
- **`is_weekend`**: 1 si el saque cae en sábado o domingo.

  **La hora exacta no se construye.** Sería la misma magnitud que la franja leída sin agrupar, y la
  hipótesis de dominio **no es lineal en la hora**: el calor de un mediodía de agosto, el ritmo
  circadiano de un saque a las diez de la noche y la audiencia de cada franja son efectos de escalón,
  no de pendiente. Emitir las dos representaciones equivaldría a entregar la misma información dos
  veces y delegar en la poda del notebook 04 una decisión que aquí se puede tomar con criterio: si lo
  que se postula son escalones, la variable que los expresa es la franja, y la hora continua solo
  añadiría una columna colineal con ella que habría que arbitrar después. La hora sí se reconstruye
  **para auditar** la franja, unas celdas más abajo, pero no viaja a la matriz.

  El día de la semana entra **solo** como `is_weekend`. La tercera categoría —el saque entre semana— ya
  viaja en el máster como `home_fat_is_midweek` / `away_fat_is_midweek`, de modo que el reparto queda
  completo sin duplicar ninguna columna.

**Geografía (2 variables, desde la tabla de estadios).**

- **`away_travel_km`**: distancia ortodrómica entre el estadio del visitante y el del local, calculada
  con la fórmula del **semiverseno**. La elección no es cosmética: la ley de cosenos esférica pierde
  precisión justamente en el caso que más abunda aquí —dos estadios muy próximos, como Sevilla y Betis a
  cuatro kilómetros—, donde el coseno del ángulo es indistinguible de uno en coma flotante.
- **`is_derby`**: 1 si ambos clubes pertenecen a la misma comunidad autónoma.

  El derbi **no** se deriva de la distancia, y es una decisión deliberada. Un umbral kilométrico sería
  una función determinista de `away_travel_km` —no aportaría información nueva— y además clasificaría
  mal los dos casos que definen el fenómeno: el derbi gallego se juega a unos 150 km y el trayecto
  Vitoria–Pamplona, que no es un derbi, a unos 90. La rivalidad local es un hecho administrativo y
  social, no métrico, así que se opera con el hecho administrativo, que es objetivo y verificable.

**La tabla de estadios se escribe a mano y se versiona en el repositorio.** Son 31 filas —los clubes que
han disputado al menos un partido de Primera en el periodo— de un dato estable; montar un scraping para
treinta filas que no cambian añadiría una dependencia de red y un modo de fallo a cambio de nada. Sí
tiene un modo de fallo propio, y silencioso: un club recién ascendido que no figure en ella deja sus dos
columnas en `NaN`, y el preprocesado le imputaría después una distancia mediana —un valor creíble y por
completo inventado—. Por eso `unknown_stadiums` se comprueba aquí y `qa_features` lo eleva a error.

**El árbitro queda fuera del alcance.** Es el tercer condicionante clásico del contexto y no se incluye
porque el dato no existe en la cadena: el fichero de LaLiga de football-data no publica la columna
`Referee` —solo la publican sus ficheros ingleses—, de modo que ni el crudo ni el máster la contienen.
Obtenerla exigiría una extracción distinta de la que hoy alimenta la estadística de partido. Se prefiere
declarar el hueco a rellenarlo con un identificador aproximado.


In [ ]:
context = build_context_features(master)
cols = [c for c in context.columns if c != 'match_key']
_cn = context_feature_names()
print(f'CONTEXTO (context.py) -> {len(cols)} columnas (match-level, sin home_/away_)\n')
for base in _cn['numeric'] + _cn['categorical']:
    s = context[base]
    if base in _cn['numeric']:
        v = s.dropna()
        det = (f'rango ({v.min():.2f}, {v.max():.2f})  media {v.mean():.2f}'
               if len(v) else 'sin valores')
        tipo = 'num'
    else:
        det = 'categorias ' + str(sorted(s.dropna().unique()))
        tipo = 'cat'
    print(f'- [{tipo}] {base:15s} cobertura {s.notna().mean():6.1%}   {det}')

# El unico modo de fallo silencioso de la geografia: un club sin estadio declarado.
print('\nclubes del master sin estadio en la tabla:', unknown_stadiums(master) or 'ninguno')
print(f'tabla de estadios: {len(stadium_table())} clubes, '
      f"{stadium_table()['region'].nunique()} comunidades autonomas")

### A.3.6-bis · ¿Miden lo que dicen medir? (auditoría descriptiva)

Las dos parejas de este bloque se apoyan en supuestos que conviene ver antes de arrastrarlas: que la
franja horaria reparte los partidos en grupos de tamaño comparable —una categoría que solo recogiera
una docena de partidos no daría para estimar nada—, y que el criterio administrativo de derbi no es
reproducible con un umbral de distancia, que es la alternativa obvia y más barata. Lo segundo se
comprueba de forma directa: si existiera un corte kilométrico equivalente, los derbis y los no derbis
ocuparían tramos separados de la escala de distancia.

In [ ]:
ctx = context.merge(master[['match_key', 'season', 'home_key', 'away_key', 'kickoff_ts']],
                    on='match_key', how='left')
ctx_dev = ctx[~ctx['season'].isin(C.HOLDOUT_SEASONS)].copy()

# La hora local se RECONSTRUYE aqui, en la celda de diagnostico, y no viaja a la
# matriz: la franja es la unica lectura del reloj que el catalogo emite. Se necesita
# para poder afirmar que las tres categorias no se solapan, que es justamente lo que
# justifica haberse quedado solo con ellas.
_h = (pd.to_datetime(ctx_dev['kickoff_ts'], unit='s', utc=True)
      .dt.tz_convert('Europe/Madrid').dt.tz_localize(None))
ctx_dev['_hora_local'] = _h.dt.hour + _h.dt.minute / 60.0

# --- 1) Reparte la franja los partidos en grupos utilizables? --------------
rep = (ctx_dev.groupby('kickoff_slot', observed=True)
       .agg(partidos=('match_key', 'size'),
            hora_min=('_hora_local', 'min'), hora_max=('_hora_local', 'max'),
            pct_finde=('is_weekend', 'mean')).round(2))
print('Reparto por franja de saque (solo desarrollo; la hora se reconstruye para auditar):')
display(rep)

# --- 2) Podria un umbral de distancia sustituir al criterio de derbi? ------
der = ctx_dev[ctx_dev['is_derby'] == 1]['away_travel_km']
nod = ctx_dev[ctx_dev['is_derby'] == 0]['away_travel_km']
print(f'derbis      n={len(der):5d}  distancia (min, mediana, max) = '
      f'({der.min():.0f}, {der.median():.0f}, {der.max():.0f}) km')
print(f'no derbis   n={len(nod):5d}  distancia (min, mediana, max) = '
      f'({nod.min():.0f}, {nod.median():.0f}, {nod.max():.0f}) km')
solapan = int(((nod >= der.min()) & (nod <= der.max())).sum())
print(f'\npartidos NO derbi dentro del rango de distancia de los derbis: {solapan} '
      f'({solapan / len(nod):.1%})')
print('=> ningun umbral kilometrico reproduce el criterio administrativo.')

# --- 3) Los extremos del desplazamiento -------------------------------------
ext = (ctx_dev.groupby(['home_key', 'away_key'], as_index=False)['away_travel_km']
       .first().sort_values('away_travel_km'))
print('\nTrayectos mas cortos y mas largos del periodo:')
display(pd.concat([ext.head(3), ext.tail(3)]).round(1))


**Qué se concluye.** Las tres franjas recogen volúmenes del mismo orden y sus rangos horarios no se
solapan: la discretización parte la variable donde dice partirla, y ninguna de las tres categorías es
un cajón residual. Esa es la comprobación que autoriza a emitir la franja **en lugar de** la hora, y no
además de ella: si los rangos se solaparan, la categoría estaría perdiendo información que solo la hora
continua podría recuperar, y habría que emitir las dos.

El solapamiento entre derbis y no derbis en la escala de distancia, en cambio, es masivo: hay trayectos
cortos que no son derbis y derbis con más kilómetros que muchos desplazamientos ordinarios, así que
ningún umbral reproduce el criterio. Las dos columnas geográficas miden cosas distintas —una el desgaste
del viaje, otra la carga de la rivalidad— y ninguna es función de la otra, que es la condición para
conservar ambas.


### A.3.7 · Fuerza latente de ataque y defensa — `strength.py`

El Elo del bloque *passthrough* resume la fuerza de un equipo en **un único escalar** y, al hacerlo,
borra la distinción entre las dos maneras de ser bueno: **marcar mucho** y **encajar poco**. Dos clubes
con idéntico rating pueden haber llegado a él por caminos opuestos —uno que gana 3-2 y otro que gana
1-0— y esa diferencia importa para un pronóstico 1X2, porque **el empate no es igual de probable** en el
encuentro de dos equipos que apenas generan que en el de dos que se desbordan mutuamente. Esta familia
estima las dos componentes por separado.

**El modelo.** Es el modelo clásico de goles del fútbol (Maher, 1982; Dixon y Coles, 1997). Cada equipo
recibe un parámetro de ataque $a$ y uno de defensa $d$; la liga, un nivel medio $\mu$ y una ventaja de
campo $h$. Los goles esperados del encuentro entre el local $i$ y el visitante $j$ son

$$\lambda_{\text{local}} = \mu \cdot a_i \cdot d_j \cdot h, \qquad
  \lambda_{\text{visitante}} = \mu \cdot a_j \cdot d_i$$

y los dos marcadores se modelan como Poisson independientes de esas medias. El ajuste es de **máxima
verosimilitud por ascenso alternado en bloques**: fijado el resto de parámetros, el óptimo de cada uno es
una razón cerrada entre **goles observados** y **goles que se esperarían de un equipo medio**, de modo
que el barrido completo se resuelve sin optimizador general y se repite hasta que nada cambia. El modelo
está sobre-parametrizado —multiplicar todos los ataques por una constante y dividir $\mu$ por ella
produce exactamente las mismas medias—, así que al cierre de cada barrido se normaliza a **media
geométrica 1** en ataques y defensas y el factor se traslada a $\mu$; sin esa restricción los parámetros
derivarían sin alterar el ajuste y las columnas dejarían de ser comparables entre fechas.

**Qué publica (5 columnas).**

- **`strength_attack_diff`** y **`strength_defence_diff`**: las dos componentes latentes, $\log a$ y
  $-\log d$, en escala logarítmica y con el criterio **positivo es mejor** —se invierte el signo de la
  defensa para que las dos se lean en la misma dirección y su lectura no obligue a recordar cuál iba al
  revés—, emitidas como **diferencia entre los dos equipos**. Cero es el equipo medio de la liga, de
  modo que $+0{,}3$ en la diferencia de ataque significa que el local marca en torno a un 35 % más de
  lo que le correspondería al rival contra ese mismo oponente. Van solo como resta porque el nivel
  absoluto de las dos ya llega al modelo por otra vía —el Elo del par, el valor de plantilla del par— y
  lo que este bloque aporta sobre aquellos es precisamente la **composición** de la ventaja, que es una
  magnitud relativa.
- **`strength_p_home`, `strength_p_draw`, `strength_p_away`**: variables **de partido**, obtenidas
  agregando por regiones la rejilla conjunta de marcadores (truncada en 10 goles por equipo y
  renormalizada, de forma que las tres suman exactamente 1 y son probabilidades legítimas).

**Los goles esperados $\lambda$ se calculan pero no se publican.** Son el paso intermedio obligado —sin
ellos no hay rejilla de marcadores y por tanto no hay terna—, pero como columna serían redundantes: la
terna 1X2 es exactamente el resumen de la distribución que $\lambda$ genera, y las dos medias por
separado no dicen del desenlace nada que la terna no diga ya. Emitirlas añadiría dos columnas por lado
y su diferencia, seis en total, deterministas dadas las que sí viajan.

**Cuatro decisiones de construcción.**

- **Sin la corrección $\tau$ de Dixon y Coles.** Ese factor corrige la probabilidad conjunta de los cuatro
  resultados de pocos goles (0-0, 1-0, 0-1, 1-1), donde la independencia se queda corta. Aquí no se
  incluye, y la omisión es deliberada: $\tau$ mejora la probabilidad de un **marcador concreto**, que no
  es lo que se pide al bloque —sus columnas son fuerzas y una terna 1X2—, y a cambio obliga a una
  optimización numérica general en lugar del ajuste cerrado, multiplicando por dos órdenes de magnitud el
  coste de un reajuste que se repite en **cada fecha** del histórico. La terna que sí se publica agrega
  regiones enteras del marcador, donde el efecto de $\tau$ es de segundo orden.
- **Memoria exponencial de media temporada.** Un partido pesa $0{,}5^{\Delta\text{días}/180}$: media
  temporada atrás vale la mitad y la temporada anterior entra completa pero pesando en torno a un cuarto,
  que es la manera de reconocer que un equipo no se reinicia cada agosto sin dejar que arrastre
  indefinidamente lo que fue. La semivida se declara como **constante del módulo** y no se ajusta contra
  el holdout: es un parámetro de construcción del dato, no de selección.
- **Encogimiento hacia la media.** Cada parámetro se estima con dos partidos ficticios de rendimiento
  exactamente medio. Sin esa referencia, un equipo que en su muestra ponderada no hubiera marcado todavía
  obtendría $a = 0$ —y por tanto un logaritmo infinito—, y uno con dos partidos afortunados obtendría una
  fuerza extrema sostenida por nada.
- **Historia mínima.** Aun con encogimiento, un equipo sin recorrido no tiene fuerza estimable, y
  devolver la media de la liga como si fuera una medición sería sustituir ignorancia por un valor creíble
  e inventado. El bloque exige **tres partidos efectivos** —la suma de los pesos, no el recuento bruto— y
  por debajo de ese umbral deja las columnas en `NaN`. Como el bloque describe un **enfrentamiento**, si
  a uno de los dos clubes le falta historia el partido entero se queda sin él: no hay medias tintas. El
  hueco lo recoge después el preprocesado con su propio indicador, `flag_no_strength`.

**Disciplina anti-fuga.** Los parámetros de un partido se estiman con los partidos disputados en fechas
**estrictamente anteriores** a la suya, de modo que el reajuste es diario: hay tantos ajustes como fechas
distintas tiene el calendario. Dos partidos del mismo día no se informan entre sí, ni un partido se
informa a sí mismo —la misma regla y el mismo corte por fecha que aplica `standings.py` a la
clasificación previa—. Las filas sin resultado no entran nunca en el ajuste, así que el módulo se
comporta igual sobre el máster jugado y sobre el máster extendido con la jornada por venir (Parte B).

**La trayectoria por equipo se expone aparte.** Como de las dos fuerzas viaja solo la resta, y de una
resta no se recupera ninguno de sus dos sumandos, el módulo publica además `strength_history(master)`:
una fila por (fecha, equipo) con su ataque y su defensa estimados. No entra en el conjunto —no está
indexada por partido— y existe para que la auditoría de la celda siguiente pueda comprobar que cada
fuerza mide lo que su nombre dice, que es una pregunta sobre equipos y no sobre enfrentamientos.

**Doble papel.** La pieza sirve para dos cosas y por eso se escribe una sola vez. Como familia de
variables aporta las cinco columnas anteriores. Como **modelo autónomo**, la terna
`strength_p_home/draw/away` es ya un pronóstico 1X2 completo —construido sin mirar el mercado y sin
ningún estimador ajustado— y se compara como tal en A.3.7-bis.


In [ ]:
strength = build_strength_features(master)
_sn = strength_feature_names()
cols = [c for c in strength.columns if c != 'match_key']
print(f'FUERZA LATENTE (strength.py) -> {len(cols)} columnas '
      f'({len(_sn["numeric"])} atributos de equipo, emitidos segun el catalogo, y '
      f'{len(_sn["match_level"])} de partido)\n')
for c in cols:
    v = strength[c].dropna()
    tipo = 'part' if c in _sn['match_level'] else 'num'
    print(f'- [{tipo}] {c:26s} cobertura {strength[c].notna().mean():6.1%}   '
          f'rango ({v.min():+.2f}, {v.max():+.2f})  media {v.mean():+.3f}')

# Lo declarado es lo emitido: los atributos de equipo salen exactamente en los lados
# que fija el catalogo, y la terna de partido tal cual. Si el modulo calculase de mas
# -lambda, por ejemplo, que se computa por dentro- esta igualdad lo delataria.
_esperadas = [n for b in _sn['numeric'] for n in CAT.nombres(b)] + list(_sn['match_level'])
assert set(cols) == set(_esperadas), set(cols) ^ set(_esperadas)

# El bloque es todo-o-nada por partido: describe un enfrentamiento, de modo que si a
# uno de los dos clubes le falta historia el encuentro entero se queda sin describir.
n_nan = strength[cols].isna().sum(axis=1)
print(f'\npartidos con el bloque completo : {(n_nan == 0).sum():5d}  ({(n_nan == 0).mean():.1%})')
print(f'partidos sin bloque             : {(n_nan == len(cols)).sum():5d}  '
      '(algun club sin recorrido suficiente)')
print(f'partidos descritos a medias     : {int(((n_nan > 0) & (n_nan < len(cols))).sum()):5d}  '
      '(debe ser 0)')

# La terna es una distribucion legitima: condicion para poder evaluarla con log-loss.
p = strength[_sn['match_level']].dropna()
print(f'\nterna 1X2  max |1 - suma| = {(p.sum(axis=1) - 1).abs().max():.2e}   '
      'probabilidad media H/X/A: ' + ' '.join(f'{x:.1%}' for x in p.mean().values))


### A.3.7-bis · ¿Mide de verdad las dos componentes? (auditoría descriptiva)

El bloque se justifica sobre tres afirmaciones que conviene ver antes de arrastrarlo. La primera es que
**ataque y defensa son cosas distintas** y que el escalar del Elo las funde: si las dos columnas fueran
una reetiquetada de la otra, o ambas una copia del Elo, el bloque no aportaría nada. La segunda es que
las fuerzas **miden lo que su nombre dice** —que el ataque estimado se corresponda con los goles que un
club acaba marcando, y la defensa con los que encaja—, y no un artefacto del ajuste. La tercera es que
la terna de probabilidad es un **pronóstico utilizable**, y no solo una transformación decorativa de las
fuerzas: eso se comprueba enfrentándola al desenlace real, contra la referencia mínima de predecir
siempre la frecuencia histórica de la liga.

Todo lo que sigue se calcula **solo sobre desarrollo**; el holdout no se mira.


In [ ]:
st = strength.merge(master[['match_key', 'date', 'season', 'home_key', 'away_key',
                            'home_elo_z', 'away_elo_z', 'ftr', 'fthg', 'ftag']],
                    on='match_key', how='left')
dev = st[~st['season'].isin(C.HOLDOUT_SEASONS)].dropna(subset=['strength_attack_diff']).copy()
print(f'partidos de desarrollo con el bloque completo: {len(dev)}\n')

# --- 1) Es el bloque una copia del Elo, o separa lo que el Elo funde? ------
# La referencia es la diferencia de fuerza TIPIFICADA, que es la que llega al
# modelo; el nivel crudo se queda en el master como columna descriptiva.
dev['elo_z_diff'] = dev['home_elo_z'] - dev['away_elo_z']
print('correlacion con la diferencia de fuerza tipificada:')
for c in ('strength_attack_diff', 'strength_defence_diff'):
    print(f'  {c:26s} r = {dev[c].corr(dev["elo_z_diff"]):+.3f}')
print(f'  y entre si: ataque vs defensa   r = '
      f'{dev["strength_attack_diff"].corr(dev["strength_defence_diff"]):+.3f}')

# --- 2) Se corresponde cada fuerza con lo que dice medir? ------------------
# Esta pregunta es sobre EQUIPOS -si el ataque estimado de un club se parece a los
# goles que ese club marca-, y la fila de partido ya no la puede responder: de los dos
# atributos el catalogo emite solo la diferencia, y de una resta no se recupera ninguno
# de los dos sumandos. La trayectoria por equipo se pide al modulo, que la expone
# justamente para esto. Debilitar la auditoria para que quepa en la matriz seria el
# camino contrario: la matriz se ha recortado, la comprobacion no.
hist = strength_history(master)
hist['date'] = pd.to_datetime(hist['date']).dt.normalize()

lados = []
for lado, gf, gc in (('home', 'fthg', 'ftag'), ('away', 'ftag', 'fthg')):
    lados.append(pd.DataFrame({
        'date': pd.to_datetime(dev['date']).dt.normalize().to_numpy(),
        'season': dev['season'].to_numpy(), 'team_key': dev[f'{lado}_key'].to_numpy(),
        'gf': dev[gf].to_numpy(), 'gc': dev[gc].to_numpy()}))
obs = (pd.concat(lados, ignore_index=True)
       .merge(hist[['date', 'team_key', 'strength_attack', 'strength_defence']],
              on=['date', 'team_key'], how='inner'))
assert len(obs) > 0, ('el cruce (fecha, equipo) contra strength_history no devolvio '
                      'filas: revise la normalizacion de fechas antes de leer nada')

ct = (obs.groupby(['season', 'team_key'])
      .agg(n=('gf', 'size'), gf=('gf', 'mean'), gc=('gc', 'mean'),
           att=('strength_attack', 'mean'), dfn=('strength_defence', 'mean')).reset_index())
ct = ct[ct['n'] >= 20]                      # club-temporadas practicamente completas
print(f'\nvalidacion sobre {len(ct)} club-temporadas '
      f'(fuerza media con la que afronta sus partidos vs goles que realmente ve):')
print(f'  ataque   vs goles MARCADOS  por partido : r = {ct["att"].corr(ct["gf"]):+.3f}')
print(f'  defensa  vs goles ENCAJADOS por partido : r = {ct["dfn"].corr(ct["gc"]):+.3f}')
print(f'  (cruzadas) ataque vs encajados          : r = {ct["att"].corr(ct["gc"]):+.3f}')
print(f'  (cruzadas) defensa vs marcados          : r = {ct["dfn"].corr(ct["gf"]):+.3f}')


In [ ]:
# --- 3) El empate: la clase que un rating escalar no sabe anunciar ---------
# La probabilidad de empate sale de la rejilla conjunta de marcadores, es decir, del
# VOLUMEN de goles que el bloque anticipa: un partido entre dos equipos que apenas
# generan concentra masa en los marcadores cortos, donde el empate abunda. Un escalar
# de fuerza no puede producir esa lectura, porque no distingue el 3-2 del 1-0. Si el
# mecanismo es real, los quintiles de `strength_p_draw` deben ordenar a la vez la
# frecuencia de empate observada (creciente) y los goles vistos (decreciente).
dev['goles_observados'] = dev['fthg'] + dev['ftag']
q = pd.qcut(dev['strength_p_draw'], 5, labels=['Q1 (mas abierto)', 'Q2', 'Q3', 'Q4',
                                               'Q5 (mas cerrado)'])
vol = (dev.groupby(q, observed=True)
       .agg(partidos=('match_key', 'size'),
            p_empate_media=('strength_p_draw', 'mean'),
            pct_empate_real=('ftr', lambda s: (s == 'D').mean()),
            goles_observados=('goles_observados', 'mean')).round(3))
display(vol)

_r_emp = dev['strength_p_draw'].corr((dev['ftr'] == 'D').astype(float))
_r_gol = dev['strength_p_draw'].corr(dev['goles_observados'])
print(f'r(p_empate, empate observado)  = {_r_emp:+.3f}')
print(f'r(p_empate, goles del partido) = {_r_gol:+.3f}')
print('(ambas bajas en valor absoluto por construccion: el desenlace de UN partido es'
      '\n casi todo ruido de Poisson; lo que el bloque aporta es la media, no el marcador)')


In [ ]:
# --- 4) La terna como pronostico 1X2 ---------------------------------------
# Referencia minima: predecir siempre la frecuencia historica de la liga en desarrollo.
Y = dev['ftr'].map({'H': 0, 'D': 1, 'A': 2}).to_numpy()
P = dev[['strength_p_home', 'strength_p_draw', 'strength_p_away']].to_numpy()
base = np.tile(np.bincount(Y, minlength=3) / len(Y), (len(Y), 1))
ll = lambda p, y: float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1))))

print(f'{"":24s}{"log-loss":>10s}{"acierto":>10s}')
print(f'{"frecuencia base":24s}{ll(base, Y):10.4f}{(base.argmax(1) == Y).mean():10.1%}')
print(f'{"modelo de goles":24s}{ll(P, Y):10.4f}{(P.argmax(1) == Y).mean():10.1%}')

# Calibracion: si la probabilidad de victoria local significa algo, la frecuencia
# observada debe seguirla decil a decil.
cal = (dev.assign(_d=pd.qcut(dev['strength_p_home'], 10, duplicates='drop'),
                  _real=(dev['ftr'] == 'H').astype(int))
       .groupby('_d', observed=True)
       .agg(partidos=('match_key', 'size'), p_media=('strength_p_home', 'mean'),
            frecuencia_real=('_real', 'mean')).round(3))
display(cal)


**Qué se concluye.** Las tres afirmaciones se sostienen, y una con matices.

*Ataque y defensa no son la misma cosa, ni son el Elo.* Cada componente correlaciona fuerte con la
diferencia de Elo —es esperable: el equipo mejor suele serlo en las dos facetas—, pero **entre sí solo
correlacionan moderadamente**, de modo que hay una cantidad sustancial de partidos en los que la
composición de la ventaja es asimétrica. Esa es exactamente la información que el escalar no puede
transportar y la razón de mantener las dos diferencias junto al Elo en lugar de en su lugar.

*Cada fuerza mide lo que su nombre dice.* En club-temporada, el ataque estimado correlaciona fuerte y
positivamente con los goles marcados por partido y la defensa, también fuerte, en el sentido correcto
con los encajados —signo negativo, dado que la defensa se publica con el criterio positivo-es-mejor—,
mientras que las correlaciones cruzadas caen sustancialmente. Las columnas son específicas, no dos
lecturas de una calidad genérica. Conviene subrayar de dónde sale esta comprobación: no de la matriz de
variables, que solo lleva las diferencias, sino de `strength_history`, la trayectoria por equipo que el
módulo publica aparte. Recortar el conjunto no ha recortado lo que se puede auditar de él.

*La lectura del empate ordena bien, con el sesgo propio de una estimación regularizada.* Los quintiles
de probabilidad de empate ordenan monótonamente la frecuencia de empate realmente observada y, en
sentido inverso, los goles vistos: los partidos que el bloque anuncia cerrados terminan con menos goles
y más empates que los que anuncia abiertos. Ese es precisamente el mecanismo por el que esta familia
puede ayudar en la clase más difícil del problema, y es una lectura que un rating escalar no puede
producir. La correlación partido a partido es baja, y debe serlo: el desenlace de un solo encuentro es
casi todo ruido de Poisson alrededor de la media, y lo que el bloque aporta al modelo es esa media.

*La terna es un pronóstico real.* Bate a la frecuencia base de la liga tanto en log-loss como en
acierto, y su calibración por deciles sigue la diagonal sin desviaciones sistemáticas. Eso valida la
terna como variable de entrada y, además, la habilita como **término de comparación** para los ensayos
de la fase 5: un pronóstico de conocimiento de dominio puro, sin mercado y sin ajuste, que cualquier
conjunto de variables que se proponga tiene que batir para justificar su complejidad.


### A.3.8 · El bloque *passthrough*: lo que ya venía listo — `columns.py`

Para que el inventario esté **completo** hay que catalogar también lo que el notebook **no** construye:
las columnas que se conocen antes del partido y entran al modelo **tal cual**, sin transformar.
`columns.py` las declara en `PREMATCH_PASSTHROUGH`. Son seis bloques —cuotas y probabilidad de
mercado, Elo, valor de plantilla, fatiga, bajas y rotación del once— producidos aguas arriba
(extractores + consolidación). No necesitan `shift` porque no describen el resultado: describen las
condiciones de partida.

Dos de ellos piden una lectura atenta. El de **bajas** no es homogéneo: junta el recuento de
ausentes con las dos medidas de cuánto duele cada ausencia —el valor de mercado que se queda fuera
y la cuota de minutos que venía disputando—, que responden a preguntas distintas y llegan por
cruces distintos. El de **rotación** es el único cuya materia prima es un hecho del partido: la
alineación se conoce después del pitido, de modo que lo que entra aquí no es el once de hoy sino la
rotación que el equipo mostró en los encuentros **anteriores**. Ese desplazamiento lo hace el
extractor, junto al dato crudo del que depende, y por eso el bloque puede pasar sin transformar
como los demás.

In [ ]:
passth = [c for c in C.PREMATCH_PASSTHROUGH if c in master.columns]
# Los grupos se leen de las listas que declara `columns.py`, no de patrones sobre
# los nombres: asi el inventario no puede desincronizarse del contrato, y una
# columna nueva sin grupo salta a la vista en vez de perderse en silencio.
grupos = {
    'mercado (probabilidad de consenso y linea de handicap)': C._MERCADO,
    'Elo (fuerza)': C._ELO,
    'valor de plantilla (prior de fuerza)': C._SQUAD,
    'fatiga (descanso / congestion)': C._FATIGUE,
    'bajas (recuento, valor economico y cuota de minutos)': C._ABSENCES,
    'rotacion del once y carga de minutos': C._LINEUPS,
}
print(f'PASSTHROUGH pre-partido (columns.py) -> {len(passth)} columnas, sin transformar\n')
visto = set()
for k, v in grupos.items():
    cols = [c for c in v if c in master.columns]
    visto |= set(cols)
    print(f'- {k}  [{len(cols)}]')
    print(f'      {cols}')
print(f'\nsin grupo declarado: {[c for c in passth if c not in visto] or "ninguna"}')


## A.4 · Prueba del `shift` (auditoría anti-fuga)

Recalculamos **a mano** la forma de un equipo y la contrastamos con lo que produce el módulo. Deben
coincidir, y la 1ª jornada debe ser `NaN` (no hay pasado que ver). Esta es la garantía de que ninguna
feature de forma incluye el partido en curso.

La prueba cubre las **dos parametrizaciones** de la memoria, porque cada una podría fugarse por su
cuenta: la ventana rectangular de tres partidos y el decaimiento exponencial, cuyos pesos se
reconstruyen aquí desde la semivida, sin llamar a `pandas`. Se añade el acumulado de temporada, que es
la tercera manera de mirar hacia atrás y la única sin horizonte.

Hay un detalle de método que la reducción del catálogo obliga a explicitar. De los treinta y cinco
atributos de forma, solo los seis del núcleo salen por lado; el resto viaja **únicamente como
diferencia**, y una diferencia no se puede contrastar contra la serie manual de un equipo. La prueba se
adapta en lugar de rebajarse: la media exponencial de puntos se compara **por lado**, porque es núcleo y
el par existe, y la ventana rectangular y el acumulado se comparan reconstruyendo la resta, para lo cual
se calcula también la serie manual del **rival** de cada partido. Es una comprobación más exigente que
la anterior, no menos: ahora la igualdad tiene que cumplirse simultáneamente en las dos series que
componen cada columna.


In [ ]:
tk, sy = 'athletic_bilbao', 2022
W = F.WINDOWS[0]                                     # unica ventana rectangular
alpha = 1.0 - 0.5 ** (1.0 / F.EWM_HALFLIFE)          # peso del ultimo partido en la ewm


# Las tres lecturas hacia atras de un equipo, a mano y partido a partido.
def _serie_manual(g: pd.DataFrame) -> pd.DataFrame:
    pts = g['points'].reset_index(drop=True)
    vent, acum, expo = [], [], []
    for i in range(len(pts)):
        prev = pts.iloc[:i]
        vent.append(prev.tail(W).mean() if i > 0 else np.nan)   # min_periods=1
        acum.append(prev.mean() if i > 0 else np.nan)
        if i == 0:
            expo.append(np.nan)
        else:
            v = prev.to_numpy()[::-1]                # del mas reciente al mas antiguo
            w = (1 - alpha) ** np.arange(len(v))     # ewm(adjust=True): pesos normalizados
            expo.append(float(np.dot(v, w) / w.sum()))
    return pd.DataFrame({'match_key': g['match_key'].to_numpy(),
                         'ventana': vent, 'acumulado': acum, 'ewm': expo})


# Serie manual de TODOS los equipos de la temporada: hace falta la del rival para poder
# reconstruir las diferencias, que es como viaja casi toda la forma.
MAN = pd.concat([_serie_manual(g.sort_values('date')).assign(team_key=t)
                 for t, g in long[long.season_year == sy].groupby('team_key')],
                ignore_index=True).set_index(['team_key', 'match_key'])

seq = long[(long.team_key == tk) & (long.season_year == sy)].sort_values('date').reset_index(drop=True)
me = MAN.loc[list(zip(seq['team_key'], seq['match_key']))]
ri = MAN.loc[list(zip(seq['opp_key'], seq['match_key']))]
sgn = np.where(seq['is_home'].to_numpy() == 1, 1.0, -1.0)      # la resta es local - visitante

fm = form.set_index('match_key').loc[seq['match_key']]
lado = np.where(seq['is_home'].to_numpy() == 1, 'home', 'away')
mod_ew = np.array([fm.iloc[k][f'{lado[k]}_form_points_ewm'] for k in range(len(seq))], dtype=float)
mod_vd = fm[f'form_points_{W}_diff'].to_numpy(dtype=float)
mod_ad = fm['cum_points_pg_diff'].to_numpy(dtype=float)

man_ew = me['ewm'].to_numpy(dtype=float)
man_vd = sgn * (me['ventana'].to_numpy(dtype=float) - ri['ventana'].to_numpy(dtype=float))
man_ad = sgn * (me['acumulado'].to_numpy(dtype=float) - ri['acumulado'].to_numpy(dtype=float))


def _eq(a, b):
    return np.allclose(np.nan_to_num(a, nan=-9), np.nan_to_num(b, nan=-9))


ok_e, ok_v, ok_a = _eq(man_ew, mod_ew), _eq(man_vd, mod_vd), _eq(man_ad, mod_ad)
print(f'{lado[0]}_form_points_ewm  == calculo manual (semivida={F.EWM_HALFLIFE:g}, por lado): {ok_e}')
print(f'form_points_{W}_diff     == resta de los dos calculos manuales:           {ok_v}')
print(f'cum_points_pg_diff      == resta de los dos acumulados manuales:         {ok_a}')

print('\n1a jornada NaN en las tres lecturas:',
      bool(pd.isna(mod_ew[0]) and pd.isna(mod_vd[0]) and pd.isna(mod_ad[0])))
assert ok_e and ok_v and ok_a
assert pd.isna(mod_ew[0]) and pd.isna(mod_vd[0]) and pd.isna(mod_ad[0])


## A.5 · Ensamblado: pre-partido + todas las familias + objetivo + split

`build_features` une el bloque passthrough con **las siete familias construidas** (forma, mercado, h2h,
clasificación y trascendencia, derivadas de precio, contexto y fuerza latente), deriva el objetivo 1X2
(`y`, `y_num`) desde `ftr`, y marca `split`: la
temporada de `HOLDOUT_SEASONS` queda como **`holdout` sellado** (no se mira para elegir features ni
hiperparámetros, para evitar el *leakage de selección*). Esta es la tabla de entrenamiento final.

In [ ]:
feat = build_features(master)
print('features_matches:', feat.shape, '| match_key unico:', feat.match_key.is_unique)
print('split:', dict(feat.split.value_counts()), '| holdout sellado:', C.HOLDOUT_SEASONS)
print('objetivo:', dict(feat.y.value_counts()))
feat[['match_key','season','split','y','y_num','home_elo_z','away_elo_z',
      'form_points_ewm_diff','cum_points_pg_diff','mkt_p_home']].head(4)


## A.6 · Procedencia y cobertura: de dónde sale cada columna y desde cuándo existe

El conjunto que acaba de ensamblarse ronda las ciento cincuenta columnas —unas ciento diez de variable
más la treintena de servicio que viaja para poder liquidar apuestas y comparar contra el mercado, y que
nunca entra al modelo— construidas sobre seis proveedores distintos. Defender ese conjunto ante un
tribunal exige poder responder por escrito a dos preguntas, y las dos se responden aquí con artefactos y
no con afirmaciones.

**¿De dónde sale cada columna?** No basta con contestar a nivel de bloque, porque dentro de una misma
familia conviven procedencias distintas: `form_npxg_ewm_diff` viene de Understat, `form_rating_ewm_diff`
de SofaScore, `form_points_ewm_diff` de football-data y `form_opp_elo_z_ewm_diff` de ClubElo, y las cuatro
son «forma». La trazabilidad se resuelve por eso a nivel de **atributo** —la magnitud que se mide— y no
de módulo ni de familia, que es la unidad a la que la procedencia es constante.

**¿Desde cuándo existe?** Las fuentes no arrancan a la vez: la disponibilidad y el valor económico de
las bajas empiezan en 2017/18 y las cuotas de cierre no aparecen hasta 2019/20. Un modelo que ignore
ese hecho entrena sobre temporadas en las que media docena de columnas son, en realidad, el valor que
les puso la imputación, y la métrica resultante describe un sistema que nunca existió.

Lo importante del diseño es que **el contrato se genera, no se transcribe**. Las filas de la tabla de
procedencia se construyen a partir de las mismas constantes que usan los módulos de variables para
emitir sus columnas —las del catálogo, que es también quien decide en qué lados sale cada atributo—, y
una verificación interna comprueba que los nombres generados coinciden exactamente con los que cada
módulo declara en su `*_feature_names()`. La consecuencia es la deseada: añadir un atributo a `form.py`
sin declarar su procedencia no produce una tabla incompleta —el modo de fallo silencioso de toda
documentación escrita a mano— sino un error inmediato. Lo único escrito a mano es el conocimiento
irreducible: qué proveedor mide cada magnitud.


In [ ]:
lin = feature_lineage()
print('procedencia:', lin.shape, '| bloques:', lin.bloque.nunique(),
      '| atributos:', lin.atributo.nunique())

print('\ncolumnas por proveedor (una columna puede depender de varios):')
_prov = {}
for fila in lin.proveedores:
    for p in fila.split(', '):
        _prov[p] = _prov.get(p, 0) + 1
for k in sorted(_prov, key=lambda x: -_prov[x]):
    print(f'  {k:16s}: {_prov[k]:>3}')

print('\npor naturaleza del dato:', dict(lin.naturaleza.value_counts()))
print('por regimen de informacion:', dict(lin.regimen.value_counts()))

# La comprobacion que convierte la tabla en un contrato: igualdad de conjuntos
# entre lo que el set contiene y lo que la tabla declara, en los dos sentidos.
_p = check_lineage(feat)
print('\ncolumnas del set sin procedencia declarada:', len(_p['sin_procedencia']))
print('columnas declaradas que faltan en el set   :', len(_p['declaradas_ausentes']),
      _p['declaradas_ausentes'][:6])

# Una misma familia, cuatro proveedores: el ejemplo que justifica resolver la
# procedencia por atributo y no por modulo.
lin[lin.columna.str.startswith('home_form_')].drop_duplicates('atributo')[
    ['columna', 'atributo', 'fuentes', 'proveedores', 'naturaleza']].head(10)

In [ ]:
cov = coverage_matrix(master, feat)
por_temp = coverage_by_season(cov)
_f = [c for c in cov.columns if c.startswith('fuente_')]

print('primera temporada con dato de cada fuente, y cobertura en su rango activo:')
for c in _f:
    activo = por_temp.loc[por_temp[c] > 0, 'season']
    ini = activo.iloc[0] if len(activo) else '—'
    med = por_temp.loc[por_temp[c] > 0, c].mean() if len(activo) else float('nan')
    print(f'  {c[7:]:20s}: desde {str(ini):<8} {med:6.1%}')

por_temp[['season', 'partidos'] + _f].round(3)

In [ ]:
# El contraste que justifica medir la fuente sobre el MASTER y el bloque sobre las
# VARIABLES: en la primera jornada de cada equipo la forma esta vacia porque no hay
# partidos anteriores que promediar, no porque la fuente haya fallado. Medida sobre
# las variables, esa propiedad deseada del diseno antifuga se contaria como carencia.
_pri = (feat.home_matches_prev == 0) | (feat.away_matches_prev == 0)
print(f'partidos con algun equipo estrenandose: {int(_pri.sum())} de {len(feat)}')
for _b in ('bloque_forma', 'bloque_mercado', 'bloque_contexto'):
    if _b in cov.columns:
        print(f'  {_b[7:]:12s}: calculable en {cov.loc[_pri.values, _b].mean():5.1%} '
              f'de esos partidos, {cov.loc[~_pri.values, _b].mean():5.1%} del resto')

print('\ncobertura de FUENTE en esos mismos partidos (no depende del arranque en frio):')
for _c in ('fuente_resultado', 'fuente_elo', 'fuente_xg'):
    if _c in cov.columns:
        print(f'  {_c[7:]:12s}: {cov.loc[_pri.values, _c].mean():5.1%} / '
              f'{cov.loc[~_pri.values, _c].mean():5.1%}')

# El registro de fuentes cierra el circulo: cada bloque, con su extractor y el
# artefacto que publica. Es la tabla que la memoria cita al describir los datos.
source_registry()[['fuente', 'proveedor', 'extractor', 'artefacto', 'granularidad']]

**Qué se concluye.** Las tres lecturas dicen cosas distintas y por eso se publican por separado.

La tabla de procedencia cierra el circuito documental: toda columna del conjunto tiene un origen
declarado y toda procedencia declarada tiene columna, de modo que el anexo de la memoria y el
conjunto que entra al modelo no pueden divergir. Que la verificación sea una igualdad de conjuntos
—y no una inclusión— es deliberado: la dirección que suele olvidarse es la segunda, y es la más
grave, porque una columna declarada y ausente no señala un descuido de documentación sino que una
fuente no llegó al maestro y su bloque se cayó en silencio.

La cobertura por temporada convierte en tabla lo que de otro modo sería una afirmación: cada fuente
tiene un primer año y una cobertura dentro de su rango activo, y esas fechas son las que justifican
que el conjunto arranque en 2016/17 y que el régimen retrospectivo se estudie aparte. El contraste
final aísla el punto que hace útil separar las dos mediciones: en los partidos donde un equipo se
estrena, la cobertura de bloque cae mientras la de fuente se mantiene. Ese patrón —fuente presente,
bloque no calculable— es la firma del arranque en frío de una ventana móvil, y es exactamente lo que
distingue el comportamiento correcto del diseño antifuga de un fallo en la cadena de datos.

Ambas matrices se publican como artefactos (`feature_lineage.parquet` y `coverage_matrix.parquet`)
en la misma pasada que escribe el conjunto de variables, porque es la única que sostiene a la vez el
maestro y las variables, que es lo que la cobertura necesita leer.

## A.7 · Verificación de las variables: bancos offline e invariantes duros

El conjunto ya está ensamblado y su procedencia declarada. Falta responder a la pregunta que un
tribunal hará antes que ninguna otra: **cómo se sabe que cada familia calcula lo que dice calcular**.
La respuesta tiene dos niveles, y son complementarios porque fallan por motivos distintos.

**Nivel 1 — bancos offline, uno por módulo.** Cada familia de variables tiene junto a ella una batería
que la ejerce con datos sintéticos y valores esperados calculados a mano, sin red y sin artefactos. Un
banco no comprueba que el conjunto real esté bien: comprueba que la *lógica* es la que se ha descrito
en esta memoria, y por eso puede ejecutarse en cualquier máquina y en la integración continua.

```
MÓDULO                BANCO                    QUÉ FIJA
──────────────────────────────────────────────────────────────────────────────────
teamlong.py           test_teamlong.py         el espejo defensivo: en la fila de un equipo,
                                               `xga` es el `xg` del rival, nunca el propio
form.py               test_form.py             las dos parametrizaciones de la memoria,
                                               el `shift(1)` y lo que NO se construye
market_form.py        test_market_form.py      el cociente rendimiento/expectativa y su escala
h2h.py                test_h2h.py              identidad del par dirigido y memoria entre temporadas
standings.py          test_standings.py        clasificación reconstruida y trascendencia
prematch_derived.py   test_prematch_derived.py estructura del precio y movimiento de línea
context.py            test_context.py          calendario, geografía y franja de saque
strength.py           test_strength.py         las dos componentes latentes y la terna 1X2
catalogo.py           test_catalogo.py         que lo no declarado no se construye
lineage.py            test_lineage.py          el contrato de procedencia
columns.py            test_regimen.py          la partición semanal / retrospectivo
build_scoring.py      test_scoring_gates.py    las guardas de la ruta de inferencia
preprocess.py         test_imputacion.py       el ajuste solo sobre desarrollo
```

Tres de esas baterías merecen comentario porque fijan errores que, de producirse, darían números
perfectamente plausibles y por tanto invisibles a la inspección visual.

`test_teamlong.py` fija el **espejo defensivo**. La primitiva *team-match long* duplica cada partido en
dos filas y, en la fila de un equipo, los atributos defensivos son los ofensivos del rival. Invertir el
espejo haría que cada equipo cargara su propio ataque como defensa: la forma seguiría estando en rango,
la correlación con el resultado seguiría siendo positiva y nada saltaría.

`test_market_form.py` fija la **escala del cociente**. `relative_perf` es la única variable nacida de
una división —rendimiento observado entre expectativa del mercado—, así que su lectura depende de que
el uno sea el punto neutro: ganar siempre partiendo de una expectativa de 1.0 debe dar exactamente 1, y
ganar siempre partiendo de 0.5 debe dar exactamente 2. El banco lo comprueba con esos casos límite y
verifica además que una expectativa nula produce un hueco y no un infinito, porque un infinito
sobreviviría al preprocesado disfrazado de valor extremo.

`test_h2h.py` fija la **identidad del par dirigido**, que es la propiedad más útil de todo el bloque.
`h2h.py` agrupa por `(equipo, rival)`, de modo que las dos filas de un partido recorren la *misma*
secuencia de enfrentamientos desde lados opuestos. De ahí que la tasa de victoria del local contra ese
rival **sea** la de derrota del visitante, que las tasas de empate coincidan y que las diferencias de
goles sean opuestas. La identidad no es una curiosidad: es lo que permite detectar una clave de club
sin canonizar, que de otro modo partiría el historial de un par en dos rivalidades a medias sin
producir un solo valor fuera de rango. Nótese dónde vive ahora esa comprobación: la fila de partido ya
no lleva las tasas por lado —de las dos magnitudes de equipo el catálogo emite solo la diferencia—, así
que la identidad se formula sobre el long dirigido que `h2h_history` expone. Es la misma disciplina que
en la fuerza latente: se recorta lo que viaja al modelo, no lo que se puede auditar.

**Nivel 2 — invariantes duros sobre el conjunto ya construido.** Un banco puede pasar y el artefacto
estar mal si una fuente llegó incompleta o si una variable nueva entró sin declarar. `qa_features.py`
cierra ese hueco con diez comprobaciones que **rompen** la cadena, de modo que un conjunto defectuoso
no puede llegar al modelo:

```
 1  antifuga        ninguna columna del propio partido (goles, métricas crudas)
 2  unicidad        una fila por match_key
 3  holdout         las temporadas selladas, todas y solo ellas, en split holdout
 4  shift           en la 1ª jornada de cada equipo toda variable de forma es nula
 5  rangos          probabilidades y tasas en [0,1], margen de mercado, zonas de
                    tabla, franja horaria e indicadores binarios
 6  head-to-head    las tasas del historial no son íntegramente nulas; la igualdad
                    entre lados de la tasa del par la exige `h2h.py` al construir
 7  contexto        todo club tiene estadio y el sello de saque no es una variable
 8  fuerza latente  la terna 1X2 suma 1 y el bloque aparece o falta entero, porque
                    describe un enfrentamiento y no a un equipo
 9  procedencia     igualdad de conjuntos entre columnas del set y declaradas
10  mercado         el rendimiento relativo es un cociente definido y no negativo, y
                    los arrastres psicológicos son indicadores binarios
```

La distinción entre los dos niveles es la misma que separa a los módulos de los notebooks: el banco
prueba la definición, el invariante prueba el artefacto. La celda siguiente ejecuta el segundo nivel
sobre el conjunto recién ensamblado y, a continuación, comprueba en vivo la identidad del par dirigido
sobre datos reales.


In [ ]:
# QA duro sobre el conjunto recien ensamblado. Se ejecuta aqui, y no solo desde el
# orquestador, porque el notebook debe mostrar el resultado y no afirmarlo: lo que se
# defiende en la memoria es la salida ejecutada, no la existencia del modulo.
from qa_features import run_qa
from h2h import h2h_history

run_qa(feat)

# Identidad del par dirigido, medida sobre datos reales. Es la comprobacion del
# invariante 6 que un banco sintetico no puede dar: alli el par se construye a mano y
# esta canonizado por definicion; aqui las claves salen de siete proveedores distintos.
# La fila de partido ya no la contiene -de las dos magnitudes de equipo el catalogo
# emite solo la diferencia-, asi que se pide al modulo el long dirigido, que es donde
# la identidad es siquiera formulable.
hh = h2h_history(master)
_c = ['win_rate_ewm', 'draw_rate_ewm', 'gd_ewm']
loc = hh.loc[hh['is_home'] == 1].set_index('match_key')[_c]
vis = hh.loc[hh['is_home'] == 0].set_index('match_key')[_c]
par = loc.join(vis, lsuffix='_loc', rsuffix='_vis')

print('\nidentidad del par dirigido (maxima discrepancia observada):')
_e = par[['draw_rate_ewm_loc', 'draw_rate_ewm_vis']].dropna()
print(f'  tasa de empate identica en los dos lados : '
      f'{float((_e.iloc[:, 0] - _e.iloc[:, 1]).abs().max()):.2e}  sobre {len(_e)} partidos')
_g = par[['gd_ewm_loc', 'gd_ewm_vis']].dropna()
print(f'  diferencia de goles exactamente opuesta  : '
      f'{float((_g.iloc[:, 0] + _g.iloc[:, 1]).abs().max()):.2e}  sobre {len(_g)} partidos')

# El simplex, en la forma que el bloque permite formular. La tasa de derrota no se
# construye porque es la de victoria del rival: de ahi que la distribucion del
# historial se lea cruzando los dos lados y no dentro de uno solo. Vale tambien donde
# el relleno neutro (1/3, 1/3, 1/3) sustituye a un enfrentamiento que nunca ocurrio.
_s = par[['win_rate_ewm_loc', 'win_rate_ewm_vis', 'draw_rate_ewm_loc']].dropna()
print(f'  win(loc) + win(vis) + empate == 1        : '
      f'{float((_s.sum(axis=1) - 1).abs().max()):.2e}  sobre {len(_s)} partidos')

# Y el puente entre el accesorio y la matriz: lo que viaja al modelo es la resta de las
# dos tasas de victoria y la tasa de empate del par, y tienen que salir de aqui.
_p = par.join(feat.set_index('match_key')[['h2h_win_rate_ewm_diff', 'h2h_draw_rate_ewm']],
              how='inner').dropna(subset=['h2h_win_rate_ewm_diff'])
print(f'\n  h2h_win_rate_ewm_diff == win(loc) - win(vis) : '
      f'{float((_p["h2h_win_rate_ewm_diff"] - (_p["win_rate_ewm_loc"] - _p["win_rate_ewm_vis"])).abs().max()):.2e}')
print(f'  h2h_draw_rate_ewm     == empate del par      : '
      f'{float((_p["h2h_draw_rate_ewm"] - _p["draw_rate_ewm_loc"]).abs().max()):.2e}')


# Parte B — Inferencia: `features_upcoming.parquet`

Ahora **predecimos**. Para puntuar una jornada nueva hacen falta las mismas variables (forma, Elo,
valor de plantilla, cuotas, **fatiga, bajas y rotación del once**) calculadas para partidos **sin
resultado**. La regla de
oro sigue siendo la misma —la feature de un partido futuro solo puede mirar **hacia atrás**—, así que
reutilizamos `build_features` sin tocarlo.

### La idea central: el *máster extendido*
En lugar de duplicar la lógica, se construyen filas de partido futuro compatibles con el esquema del
máster (todo lo *post-partido* = `NaN`, porque todavía no existe) y se **concatenan** al máster jugado:

```
   máster jugado (post-partido real)
 + filas futuras   (jornada N, post-partido = NaN)
 ──────────────────────────────────────
   máster extendido  →  build_features (LA MISMA de la Parte A)  →  se filtran las filas futuras
```

Como cada feature mira solo partidos anteriores del equipo, **las filas jugadas no se alteran** y las
futuras heredan su historial. Después se marcan `split='predict'` y su objetivo `y` va nulo.

> **Detalle delicado 1 — el orden importa.** Primero se *inyectan* las filas futuras (con sus datos
> pre-partido conocidos y el resto en `NaN`); *después* se calcula la forma sobre todo junto. Si se
> calculara la forma antes de inyectar, los futuros no tendrían forma. Inyectar-primero es lo que
> permite que el mismo cálculo del entrenamiento les asigne su forma histórica.

## B.1 ¿Qué partidos se predicen, y cómo sé que están *todos*?

La fuente **principal** de "qué partidos vienen" es el extractor `footballdata.py upcoming()` (se corre
en local, necesita red): descarga el `fixtures.csv` de football-data, filtra LaLiga (SP1) y lo normaliza
a **`footballdata_upcoming.parquet`** — claves (`match_key`, `home_key`, `away_key`, fecha), **cuotas de
apertura** y `mkt_p_*` (probabilidad implícita del mercado). No trae resultado ni cuotas de cierre: aún
no existen.

**Principal y no única, y la diferencia importa.** football-data sirve el fixture y el precio *juntos o
ninguno de los dos*, de modo que mientras no publica la jornada no hay ni fila que puntuar ni cuota con
la que hacerlo. Por eso la ventana se **siembra** además desde el calendario de SofaScore
(`build_scoring._seed_from_sofascore`), que conoce la jornada con semanas de antelación: aporta el
fixture **sin cuota** y marca su procedencia en `fixture_src`. football-data **manda cuando tiene el
fixture**, porque entonces trae el precio real de las casas; la siembra solo cubre lo que a él le falta.

La siembra se recorta a la **primera** jornada que viene (`_solo_proxima_jornada`, hueco de calendario
de 3 días, el mismo criterio que `protocol._GAP_DIAS`). El recorte no es cosmético: `fixtures.csv` se
autolimita a ~7 días, así que el horizonte de 10 días nunca llegaba a abarcar dos jornadas, pero
SofaScore conoce la temporada entera y sin recortar sembraría la jornada siguiente **y la de después**
—diecinueve partidos en lugar de diez—. Los nueve de más no tienen mercado abierto en ninguna casa:
teclear su cuota sería inventarla.

**El pipeline es dirigido por *fixture*, no por número de jornada.** Se predice exactamente lo que
football-data lista como próximo, menos lo que ya está en el máster jugado. Nunca se asume
"una jornada = 10 partidos". Eso, que parece un detalle, resuelve de golpe los casos raros del
calendario sin lógica especial:

| Situación | Qué pasa |
|---|---|
| Partido **aplazado** | sale de la ventana → no se predice |
| Partido **cancelado** | nunca entra → no se predice |
| Partido **adelantado** de otra jornada | si está en la ventana, se predice igual |
| **Semana reducida** (p.ej. Supercopa: dos equipos no juegan liga) | solo aparecen los partidos programados; no se inventan los que faltan |

**El punto delicado: la ventana son ~7 días.** `fixtures.csv` es una ventana *móvil* de aproximadamente
una semana. Una jornada de LaLiga se reparte de viernes a lunes. Si ejecutas el extractor **4 días
antes** del primer partido, el del lunes está a ~7-8 días vista y puede **no haber entrado todavía** en
la ventana. Y ojo con la trampa: un partido que aún no está en la ventana **no aparece en `up`** — no
se predice y el *gate* de cuotas ni se entera (solo salta para partidos presentes sin cuota). Un
partido que falta entero es invisible para el gate.

Por eso el diseño **no se apoya en una sola descarga**:
1. **Acumulación por upsert.** El extractor no sobrescribe: fusiona cada descarga en el fichero
   acumulado. La forma de trabajar es ejecutarlo **varias veces** en los días previos; cada pasada
   añade lo que ahora entra en los 7 días, y para la víspera tienes la jornada completa.
2. **Un chequeo explícito de completitud** (sección B.2) que cruza la ventana de football-data contra
   el calendario completo de SofaScore y te dice, partido a partido, qué falta y por qué.

> **Modo demostración.** Si `footballdata_upcoming.parquet` aún no está descargado, el notebook coge la
> **última jornada ya jugada**, la *disfraza* de futura (le deja solo lo conocible antes del partido:
> claves + cuotas) y usa como historia únicamente lo anterior. Además de enseñar la mecánica, es una
> **prueba de equivalencia** (B.5).

In [ ]:
UP = PROC/'footballdata_upcoming.parquet'
if UP.exists():
    DEMO = False
    up = pd.read_parquet(UP)
    played = master
    print('MODO REAL - footballdata_upcoming.parquet:', up.shape)
else:
    DEMO = True
    cnt = master['date'].value_counts()
    full = sorted(d for d, n in cnt.items() if n >= 5)   # fechas de jornada completa
    demo_date = full[-1]                                 # ultima jornada completa jugada
    target = master[master['date'] == demo_date].copy()
    played = master[master['date'] < demo_date].copy()   # SOLO historia previa
    keep = (['match_key','season','season_year','date','home_team','away_team','home_key','away_key']
            + [c for c in target.columns if c.startswith(('odds_','mkt_p_'))])
    up = target[keep].copy()                             # jornada disfrazada de futura
    print(f'MODO DEMO - no hay upcoming descargado; uso la ultima jornada jugada ({demo_date})')
    print(f'  partidos a puntuar: {len(up)}   | historia previa: {len(played)}')
up[['match_key','date','home_team','away_team','odds_avg_h','odds_avg_d','odds_avg_a','mkt_p_home']].head()

## B.2 ¿Están *todos* los partidos de la jornada? — diagnóstico de completitud

`build_scoring` imprime un diagnóstico que **cruza la ventana de football-data (lo que se va a
predecir) contra el calendario de SofaScore** (`sofascore_team_fixtures.csv`, la única fuente con
`status`: *notstarted / postponed / canceled*, y que llega hasta fin de temporada). Para cada partido
informa de:

- los que **se predicen** y si SofaScore los confirma como programados o marca un status raro,
- los que SofaScore **programa pero football-data aún no trae**: no quedan fuera. La siembra los
  mete en la ventana sin cuota y el *gate* los pone en la plantilla, de modo que lo que el diagnóstico
  informa de ellos es su `fixture_src`, y no un hueco pendiente de que alguien reaccione,
- los **aplazados / cancelados** de la ventana (esperado que falten),
- **conflictos** football-data ↔ SofaScore.

No usa número de jornada (no existe en los fixtures futuros de ninguna fuente); el cruce por *status*
hace el mismo trabajo sin depender de un conteo nominal.

> **Un matiz honesto:** el cruce vale lo que valga la última descarga de `sofascore_team_fixtures.csv`.
> Refresca también ese calendario antes de puntuar, o estarás comparando contra una foto vieja.
>
> En **modo demo** los partidos aparecerán con `status='finished'` (pedimos prestada una jornada ya
> jugada); en scoring real saldrían como `notstarted`. La mecánica del cruce es idéntica.

In [ ]:
# 'hoy' simulado: unos dias antes del primer fixture, para que la ventana tenga sentido
fixtures = BS._try_read_csv(BS.FIXTURES, 'team_fixtures')
today_sim = pd.to_datetime(up['date']).min().normalize() - pd.Timedelta(days=3)
BS._completeness_report(up.copy(), fixtures, played, today_sim)

## B.3 Filas de partido futuro compatibles con el máster

`build_scoring._future_master_rows` arma, para cada fixture, una fila con **todas** las columnas del
máster pero con lo *post-partido* en `NaN` (goles, `sofa_*`, xG, rating, `ftr`...). Luego engancha
**todos los bloques pre-partido que se pueden conocer**:

- **Elo** → `spine.add_clubelo` (merge_asof: último Elo vigente ≤ fecha).
- **Valor de plantilla** → `squadvalue.reduce_squad_value` (por `season_year`).
- **Fatiga** → `reducers.reduce_fatigue`. El calendario de SofaScore ya trae los fixtures futuros, así
  que el descanso y la congestión multi-competición de un partido por jugar **sí** se conocen mirando
  hacia atrás.
- **Bajas, recuento** → `reducers.reduce_availability` (recuentos de *missingPlayers*). Solo existen
  cuando SofaScore ya publicó el parte del partido; si aún no hay alineaciones, quedan `NaN` (hueco
  honesto).
- **Bajas, ponderadas** → `consolidate.build_absences`, con el **mismo** cálculo que en el máster: el
  mismo desfase `t-1` en los valores de Transfermarkt y el mismo instante de corte —00:00 UTC del día
  del partido— para la cuota de minutos. No es una versión simplificada para inferencia: es la misma
  función, y por eso la prueba de equivalencia de B.5 puede exigir identidad bit a bit.
- **Rotación y carga de minutos** → `reducers.reduce_lineups`. Es el único bloque de la familia
  SofaScore que **no** depende de que la fuente publique nada del partido a predecir: se calcula
  mirando solo hacia atrás, así que basta con que el extractor esté al día. Si falta, lo que ha
  fallado es el refresco, no la publicación.

**El puente que lo hace posible.** Fatiga, bajas y rotación se identifican por `event_id`. El puente *canónico*
de consolidación se construye desde los partidos *jugados*, así que no conoce los `event_id` futuros.
Aquí se usa un **puente futuro-consciente** (`_fixtures_event2key`) que sale del crudo
`sofascore_team_fixtures.csv` —el único que trae los fixtures venideros— construyendo
`match_key = fecha|local|visitante`.

In [ ]:
tv  = pd.read_parquet(PROC/'team_squad_value.parquet')
elo = pd.read_parquet(PROC/'clubelo_elo.parquet')
fat = BS._try_read(BS.FATIGUE, 'team_fatigue')
av  = BS._try_read(BS.AVAIL,  'team_availability')
missing = BS._try_read_csv(BS.MISSING_RAW, 'bajas (valor EUR)')
pv  = BS._try_read(BS.PLAYERVAL,  'valores de jugador')
lin_art = BS._try_read(BS.LINEUPS,   'rotacion y minutos')
pm  = BS._try_read(BS.PLAYERMIN, 'bajas (rol: minutos)')

# Todos los artefactos entran por la MISMA via que en produccion: `_try_read`
# aplica la politica de bloques (un artefacto exigible que falte corta la pasada
# con su motivo) y Elo y valor de plantilla se recalculan desde su parquet, no se
# leen ya consolidados del master. Asi lo que se audita aqui es exactamente la
# ruta que recorre el scoring semanal, con todos sus enganches y no con un
# subconjunto que dejaria fuera justo los bloques mas delicados.
fut = BS._future_master_rows(up, played, elo, tv, fat, av, fixtures,
                             missing=missing, pv=pv, lin=lin_art, pm=pm)
print('filas futuras:', fut.shape)
print('\npre-partido enganchado vs post-partido en NaN (goles/xg/rating):')
fut[['match_key','home_elo_z','home_squad_value','home_fat_rest_days','home_n_missing',
     'home_out_value','home_out_minshare','home_xi_continuity',
     'fthg','ftag','home_xg','home_sofa_rating']].head()


## B.4 Máster extendido, *gate* de cuotas y `build_scoring`

`build_scoring` concatena `jugados + futuros`, ejecuta **la misma `build_features` de la Parte A** sobre
la unión y devuelve solo las filas futuras con `split='predict'` y objetivo nulo. Antes de escribir nada
aplica dos controles operativos que el entrenamiento no necesita:

- **Cuotas manuales + gate estricto.** Si football-data aún no publicó la cuota de algún partido,
  `build_scoring` escribe una **plantilla CSV** (`footballdata_upcoming_manual_odds.csv`) con los
  partidos que faltan; rellenas las cuotas 1X2 que veas en tu casa y re-ejecutas. Las manuales pasan por
  la **misma** normalización de-margen que las automáticas. Si tras auto + manual **algún** partido
  sigue sin cuota, se **para** y no se escribe `features_upcoming` (la cuota es una señal demasiado
  fuerte como para predecir sin ella). Fatiga/bajas sí pueden quedar `NaN`.
- **Y lo tecleado prevalece después en el histórico**, que es la mitad sin la cual el mecanismo sería de
  un solo uso: una semana más tarde el partido se ha jugado, football-data lo publica en el CSV
  histórico con *su* precio, el máster se reconstruye y toma ese — la predicción se emitió a un precio y
  el histórico la reentrena y la liquida a otro, sin que nada avise, porque las dos cifras son
  legítimas. `promote_manual_odds` guarda lo tecleado con su sello `entered_ts` en
  `footballdata_odds_manual.csv` y `footballdata.apply_manual_odds` lo vuelve a aplicar en cada
  reproceso. Teclear una vez basta.

  Su **alcance** es una decisión con criterio y no una omisión: toca exclusivamente `odds_avg_h/d/a`, el
  canal de entrada del modelo, de donde salen `mkt_p_home/draw/away`. **No toca `odds_b365_*`,
  `odds_ps_*`, `odds_max_*` ni ninguna columna de cierre**, porque las de Bet365 son el listón contra el
  que se publica toda cifra económica (`backtest.REFERENCIA`) y una cuota escrita a mano ahí haría que
  la pregunta del trabajo se contestara contra un precio que no publicó ninguna casa; y porque cuando
  football-data publica el partido ya jugado trae la cuota de **cierre**, mejor dato que la apertura que
  se vio al teclear, de modo que pisarla degradaría el histórico en lugar de corregirlo. Lo que
  prevalece es **con qué precio predijo** el modelo; lo que se conserva es **a qué precio se liquida**.

  El límite que lo hace defendible es la regla antifuga escrita en código: la cuota se aplica solo si su
  sello es **anterior al día del partido**, de modo que el override queda confinado por construcción a
  partidos que estaban por jugarse cuando se tecleó y los bloques de validación son intocables. Como las
  dos vías son indistinguibles aguas abajo —misma normalización, mismo rechazo de márgenes imposibles—,
  la procedencia se declara en `odds_src`.
- **Log de *readiness* por partido.** Imprime, a X días del saque, qué se sabe de cada partido: fuente
  de la cuota (auto/manual/falta), Elo, valor de plantilla, fatiga y si las bajas están publicadas.
  Pensado para ejecutarlo varias veces antes de la jornada y ver cómo se va "llenando".

> **Detalle delicado 2 — `_points` NaN-safe.** Un partido futuro tiene goles `NaN`. Sin cuidado,
> `np.where(gf>ga,...)` lo trataría como derrota (0 puntos) y ese 0 espurio contaminaría la ventana
> rolling de una jornada futura posterior. `teamlong._points` deja `NaN` cuando no hay resultado, y el
> rolling lo ignora. No afecta al entrenamiento (los jugados siempre tienen goles).

In [ ]:
from build_scoring import MissingOddsError
try:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', FutureWarning)  # concat con columnas all-NA (post-partido futuro)
        scored = build_scoring(up=up, played=played, elo=elo, tv=tv,
                               fat=fat, av=av, fixtures=fixtures,
                               today=today_sim, strict_odds=True)
    print('\nfeatures_upcoming:', scored.shape, '| match_key unico:', scored['match_key'].is_unique)
    print('split:', dict(scored['split'].value_counts()), '| objetivo y nulo:', scored['y'].isna().all())
    display(scored[['match_key','date','home_team','away_team','home_elo_z',
            'form_points_ewm_diff','home_fat_rest_days','home_n_missing','mkt_p_home']].head())
except MissingOddsError as e:
    print('GATE DE CUOTAS activado - NO se escribe features_upcoming:\n')
    print(e)


## B.5 Prueba de equivalencia hacia atrás (la auditoría clave)

La forma de un partido futuro debe ser **idéntica** a la que produciría el entrenamiento si ese partido
estuviera en la tabla: misma historia previa, misma fórmula con `shift`. La comparación cubre
**también fatiga, bajas y rotación**: son *passthrough* (build_features las pasa sin tocar), así que
los valores que engancha el puente futuro-consciente deben ser idénticos a los que el máster ya tenía
almacenados para ese partido. Comparamos forma + Elo + valor + fatiga + bajas + rotación de `scored`
contra
`build_features(máster completo)` para esos mismos `match_key`. Deben coincidir **bit a bit** (salvo
`NaN`). Si coinciden, la ruta de inferencia no filtra y es consistente con el train.

In [ ]:
train = build_features(master)
tr = train[train['match_key'].isin(scored['match_key'])].set_index('match_key').sort_index()
sc = scored.set_index('match_key').sort_index()

# Las columnas de forma se piden al catalogo en vez de adivinarse por prefijo: desde que
# la mayoria sale solo como diferencia (`form_ppda_ewm_diff`), ningun rotulo de lado las
# captura, y un patron que se quedara corto haria pasar la prueba sin comparar nada.
form_cols = [c for b in form_feature_names()['numeric'] for c in CAT.nombres(b) if c in tr.columns]
sv_cols  = [c for c in C._SQUAD if c in tr.columns]
fat_cols = [c for c in C._FATIGUE if c in tr.columns]
baj_cols = [c for c in CAT.nombres('n_missing') if c in tr.columns]   # recuentos; el valor va aparte
check = [c for c in (form_cols + sv_cols + fat_cols + baj_cols) if c in sc.columns]

diffs = {}
for c in check:
    a = pd.to_numeric(sc[c], errors='coerce'); b = pd.to_numeric(tr[c], errors='coerce')
    same = np.isclose(a.fillna(0), b.fillna(0), rtol=1e-9, atol=1e-9) | (a.isna() & b.isna())
    if (~same).sum(): diffs[c] = int((~same).sum())

print(f'columnas comparadas: {len(check)} '
      f'(forma={len(form_cols)}, valor={len(sv_cols)}, fatiga={len(fat_cols)}, bajas={len(baj_cols)})')
print('discrepancias:', diffs if diffs else 'NINGUNA')
assert not diffs, diffs
print('OK EXACTO - la inferencia reproduce el entrenamiento (forma + valor + fatiga + bajas) bit a bit.')


## B.6 Qué queda `NaN` en el scoring y por qué

Con todos los bloques enganchados, en las filas a predecir ya **no** quedan `NaN` por diseño en fatiga
(el calendario de SofaScore la cubre mirando hacia atrás). Lo que puede seguir `NaN` es:

- **Bajas**, si SofaScore aún no publicó el parte del partido (alineaciones y lesiones se conocen
  tarde). En cuanto se publican, aparecen; hasta entonces, hueco honesto. Afecta por igual al
  recuento y a las dos ponderaciones, porque las tres cuelgan del mismo parte.
- **Cuota de minutos de los ausentes** (`out_minshare`, `out_minshare_max`), que además puede faltar
  por su cuenta: en las primeras jornadas de una temporada no hay denominador —ningún futbolista ha
  acumulado minutos todavía—, de modo que el sub-bloque está legítimamente vacío aunque el parte médico
  sí exista. Se vigila como bloque aparte para no atribuir ese hueco al extractor equivocado.
- **Rotación del once** (`xi_continuity`, `starters_minutes_14`, `minutes_hhi`), por la misma razón y
  con el mismo matiz: no depende de que la fuente publique el partido a predecir, sino de que el
  equipo haya jugado ya lo suficiente en la temporada.
- **Forma en las primeras jornadas** de un equipo recién ascendido, igual que en el entrenamiento.
- **Fuerza latente** (`strength_attack_diff`, `strength_defence_diff` y la terna) en esas mismas
  jornadas y por la misma razón, pero con una diferencia operativa que conviene tener presente: el
  bloque describe un **enfrentamiento**, así que basta con que a **uno** de los dos clubes le falte
  recorrido para que el partido entero se quede sin él. Con dos ascendidos estrenándose, dos de los
  diez partidos de una jornada pueden carecer del bloque sin que nada haya fallado, y por eso su *gate*
  de cobertura se declara con una tolerancia distinta de cero en lugar de exigir presencia total.
- **Estructura del precio** (`mkt_overround_open`, `mkt_spread_mean`, `sharp_div_home/draw/away`), si la
  cuota del partido llegó por la **plantilla manual**: ahí solo se teclea el precio 1X2, no el desglose
  por casa que esas variables necesitan. Es un hueco por diseño, no un fallo de la cadena de datos, y
  por eso el bloque se informa pero **no** activa el gate: lo obligatorio es tener precio, no tenerlo
  desglosado.

Todo esto se **imputará en el preprocesado posterior** con la misma política que las primeras jornadas
del train. La forma y el mercado siempre están (el mercado, por el gate, es obligatorio).

**Un matiz sobre la dureza del calendario.** `form_opp_elo_z_ewm_diff` y `form_opp_elo_z_3_diff` —la
fuerza media de los rivales a los que un equipo se ha enfrentado— son el único sub-bloque de forma que encadena
*dos* fuentes: el máster dice contra quién se jugó y ClubElo dice cuánto valía ese rival. Por eso
`build_scoring` lo vigila como un bloque aparte en su *gate* de cobertura y no mezclado con el resto de
la forma: el *gate* declara vacío un partido solo cuando le faltan **todas** las columnas del bloque,
así que fundido con la media de puntos —que casi siempre está— la ausencia completa de la dureza del
calendario quedaría enmascarada y el partido se puntuaría con un bloque entero imputado sin que nada
avisara. El mismo argumento aísla la calidad de ocasión (`form_shot_xg_mean_*`), que depende del detalle
de tiro de Understat y no del máster.


In [ ]:
# Los bloques no se recomponen aqui con patrones sobre los nombres: se leen de la
# tabla de procedencia, igual que en `qa_features` y en el informe de readiness de
# `build_scoring`. Asi la cobertura de la jornada que se va a predecir usa la misma
# particion de columnas que la del conjunto con el que se entreno, y comparar una
# con otra significa algo.
_pres = set(scored.columns)

def cov(cols, label):
    cols = [c for c in cols if c in _pres]
    m = scored[cols].notna().mean().mean() if cols else float('nan')
    print(f'  {label:32s}: {m:6.1%}  ({len(cols)} cols)')

print('cobertura por bloque analitico en las filas a predecir:')
for _b, _g in lin.groupby('bloque', sort=True):
    cov(list(_g.columna), _b)

# El desglose que importa el dia de la jornada es el de PROVEEDOR: si un extractor
# no se ha ejecutado esta semana, el hueco no se reparte por bloques analiticos, se
# concentra en todas las columnas que dependen de esa fuente.
print('\ncobertura por proveedor de datos:')
for _p in sorted({p for fila in lin.proveedores for p in fila.split(', ')}):
    cov(list(lin.loc[lin.proveedores.str.contains(_p, regex=False), 'columna']), _p)

# El bloque de fuerza es todo-o-nada por partido: interesa el recuento de partidos
# sin describir, no la media de columnas, porque una cobertura del 80 % significa
# aqui '2 de 10 partidos sin bloque' y no 'todos los partidos algo incompletos'.
_sc = [c for c in scored.columns if 'strength_' in c]
if _sc:
    _sin = scored[_sc].isna().all(axis=1).sum()
    print(f'\n  -> partidos sin fuerza latente: {_sin} de {len(scored)} '
          '(algun club sin recorrido suficiente en la categoria)')

# La cobertura de la jornada se publica ademas como artefacto, con la misma
# definicion que la del conjunto de entrenamiento. Es lo que permite auditar, sobre
# una prediccion ya emitida, con que informacion se hizo: sin ese registro, una
# jornada puntuada con la fatiga a medias es indistinguible de una puntuada con todo
# el dato, porque el preprocesado imputa el hueco y no queda sintoma visible.
_cu = scored.attrs.get('cobertura')
if _cu is not None:
    print('\ncoverage_upcoming:', _cu.shape,
          '| fuentes completas por partido:', dict(_cu.n_fuentes.value_counts()))

## Cierre

Los dos sets se escriben con los módulos, **no** desde el notebook (el notebook expone y audita; los
módulos producen):

```bash
# Parte A - entrenamiento
python scripts/features/build_features.py          # construye features_matches.parquet
python scripts/features/qa_features.py             # audita anti-fuga

# Parte B - inferencia (antes de cada jornada)
python scripts/extractors/footballdata.py upcoming # 1) descarga proximos partidos (local, red)
python scripts/features/build_scoring.py           # 2) genera features_upcoming.parquet
python scripts/features/build_scoring.py --dry     #    (solo reporta, no escribe)
```

**Cómo ejecutarlo bien el día de la jornada:**
- **Refresca el calendario de SofaScore** antes de puntuar. No es solo para que el chequeo de
  completitud (B.2) compare contra una foto actual: es también la fuente de la que se siembra la ventana
  cuando football-data va con retraso, así que una foto vieja del calendario deja la jornada fuera.
- **Una sola pasada de `upcoming` basta para que la jornada entre.** Acumular por upsert es lo que hace
  el extractor y conviene igual, porque cada pasada trae precios reales de las casas en lugar de huecos
  que teclear; pero el partido que su ventana de 7 días no alcanza entra por la siembra, con la cuota
  pendiente de teclado en vez de no entrar.
- Si el **gate de cuotas** para el proceso, es a propósito: rellena
  `footballdata_upcoming_manual_odds.csv` y re-ejecuta. Desde la rutina semanal se reanuda sin repetir
  la cadena, con `python scripts/rutina_semanal.py --desde predict`.
- **Teclea solo lo que hayas visto en una casa.** El gate acepta cualquier terna 1X2 plausible y no
  puede distinguir un precio real de uno inventado; lo que sí distingue es la procedencia, y por eso la
  guarda en `odds_src` y en el registro con su sello.

**Garantía demostrada:** la ruta de inferencia usa exactamente el mismo código de features que el
entrenamiento y solo mira hacia atrás — la prueba de equivalencia (B.5) lo confirma bit a bit,
también para fatiga, bajas y rotación.

**Aviso multi-jornada:** si se puntúan 2+ jornadas futuras a la vez, el acumulado de temporada
(`cum_*_pg`) de la 2ª jornada en adelante sale `NaN` (el `cumsum` arrastra el `NaN` de la jornada
futura anterior); la forma *rolling* sí sale bien porque ignora `NaN`. Lo habitual es puntuar una
jornada cada vez.

Con esto, la etapa de creación de features queda cerrada y auditada para train y para scoring. Los
`NaN` por diseño (forma y rotación en las primeras jornadas, bajas sin publicar) se resuelven en la
etapa siguiente:
el **preprocesado**, implementado en `scripts/features/preprocess.py` y orquestado por
`scripts/features/build_preprocess.py`. Su análisis y su justificación —qué se imputa y con qué
criterio, qué se descarta y qué se transforma— ocupan el notebook `04_ANALISIS_Y_PREPROCESADO.ipynb`.
Allí los parámetros se ajustan **solo sobre desarrollo**, de modo que la garantía antifuga que aquí se
demuestra a nivel de features se prolonga hasta la matriz que entra al modelo.

**Dónde acaba esta ruta.** `features_upcoming.parquet` no es todavía lo que puntúa el
modelo: es la jornada por jugar expresada en el mismo lenguaje de variables que el
conjunto de entrenamiento. Falta aplicarle el estado de preprocesado congelado —el que
la Parte D del notebook 04 aprende sobre desarrollo y `build_preprocess.py` serializa—,
y de eso se encarga `scripts/features/build_predict.py`, que escribe
`features_predict.parquet` con exactamente las mismas columnas y en exactamente el
mismo orden que la matriz de entrenamiento. Esa identidad de esquema no es una
comodidad: es el tercer invariante que audita `scripts/models/qa_models.py`, porque un
reordenamiento de columnas no produce ningún error visible —el estimador aplicaría cada
coeficiente a la columna equivocada y devolvería probabilidades perfectamente
plausibles— y solo se manifestaría como una degradación silenciosa que nadie vuelve a
mirar.

La cadena, por tanto, encadena tres notebooks: aquí se fabrica la variable y se
demuestra que mira solo hacia atrás; en el 04 se decide cómo se limpia, se imputa y se
transforma; y en el `05_MODELADO_Y_EVALUACION.ipynb` se ensayan a mano conjuntos de
variables sobre las cuatro familias, bajo el protocolo de validación temporal y con el
apostante de referencia siempre al lado; el conjunto que se elige se cierra en
`build_models.py` y la temporada sellada se lee una única vez, al final.